<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎞️ FlashVSR+ Video Upscaler</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU T4 x2 Edition - Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0;'>One-step diffusion video upscaling to 720p, 1080p or 1440p | FP16 DiT split across both T4s | Powered by FlashVSR+ Turbo & Base</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-GPU%20T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

### What is FlashVSR+?

**FlashVSR+** (Shanghai AI Lab, CUHK, Tsinghua) turns low-resolution or blurry video into sharp high-resolution video in a single diffusion step. It streams the video in 16-frame chunks with a sparse attention cache, so it keeps details stable from frame to frame and handles long clips without running out of memory.

| Feature | Details |
|----|----|
| Models | **Turbo** (hybrid sliding attention, fastest) and **Base** (sparse attention in every layer, finest detail): a 1.3B DiT built on Wan2.1, one denoising step |
| Decoder | FlashVSR+'s tiny conditional decoder, which also sees the input frames |
| Output | 720p, 1080p, 1440p or 2x / 3x / 4x, exact size (no cropping), H.264 with the original audio, saved to `/kaggle/working/outputs` (Kaggle Output panel) |
| Before / after | A comparison player under the result: drag the divider between the bicubic-resized input and FlashVSR+, with synced play / pause, frame-by-frame stepping, scrubbing and fullscreen |
| Long videos | Streamed in and out chunk by chunk: RAM and VRAM stay flat however long the clip is |
| Fast motion | Optional Large-motion modes (2x / 4x slower) keep fine detail in fast action |

### ⚡ What makes it fast on T4

| Change | Why it helps on T4 |
|----|----|
| **FP16** instead of BF16 | T4 has no BF16 tensor cores, so the release's BF16 settings would run on a slow emulated path |
| **Both GPUs work at once** | GPU 0 runs the input projection and the first DiT blocks of the next chunk while GPU 1 runs the last blocks and the decoder of the current one. The split point is measured on each new resolution and moved until both GPUs take the same time |
| **Sparse attention kernel tuned for T4** | FlashVSR+'s Triton kernel uses 128x128 tiles, which need more shared memory than a T4 has. At startup each GPU compiles smaller tiles, checks them against an exact reference and keeps the fastest |
| **FP32 RoPE** with cached angles | The release computes RoPE in FP64, which a T4 runs at 1/32 speed |
| **Streaming I/O** | CPU threads resize the frames ahead of time, frames stay 8-bit until they reach the GPU, and every decoded chunk goes straight to ffmpeg |

Same weights, one-step sampler, attention masks, caches and decoder as the official pipeline. What changed is the number format (FP16) and how the work is spread over the two GPUs: each DiT block keeps its attention cache on the GPU that runs it, so splitting the model does not change the result.

### Quick Start
1. **Settings → Accelerator → GPU T4 x2**, and **turn on Internet** in the Settings sidebar
2. Run all cells. **Step 1** installs and downloads ~14 GB of weights (Turbo + Base). **Step 2** loads the model and starts the UI (~2 min)
3. Open the **Cloudflare** link or the **Gradio** share link printed at the bottom of Step 2
4. Upload a video, pick the **output size** (1080p by default) and press **Upscale Video**. Tip: set **Seconds to upscale** to 3-5 for a quick look first
5. Check the result in the **Before / after** player: drag the divider, pause, and step frame by frame with ← →
6. After each video, Step 2 prints a **📊 Speed** line: frames per second, seconds per frame, and how long each GPU spends per chunk

> 💡 **Best results:** FlashVSR+ is trained for **4x** upscaling of low-resolution video (e.g. 270p → 1080p) and works well from 2x to 4x. A T4 handles up to **1440p** output; 4K needs a 24 GB GPU.

> 💡 **Save the downloads:** after the first run, create a Kaggle Dataset from `/kaggle/tmp/flashvsr_models` and attach it with **Add Input**. Step 1 finds the big files there and skips the download.

---
## ⚙️ Step 1: Setup + model downloads

Fetches the FlashVSR+ code (pinned to a tested commit) and the few Python packages it needs on top of Kaggle's own PyTorch, then downloads the weights to `/kaggle/tmp/flashvsr_models` with one progress bar:
- **Turbo** DiT (5.7 GB) + LQ projection (1.2 GB)
- **Base** DiT (5.7 GB) + LQ projection (1.2 GB): set `GET_BASE_MODEL = False` below to skip it
- **TC decoder** (0.2 GB)

In [ ]:
# Step 1: Environment, pinned FlashVSR+ code, dependencies and model downloads (~14 GB)
import os
import sys
import time
import json
import shutil
import subprocess
import threading
import psutil
from concurrent.futures import ThreadPoolExecutor


def stop_old_engine():
    # A previous run's engine (or one left behind by a kernel restart) still holds both GPUs and RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_flashvsr.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} leftover engine process(es) from an earlier run: GPU memory and RAM are free again")


stop_old_engine()
T_START = time.time()

# ---- Settings ----
GET_BASE_MODEL = True  # False: Turbo only (saves 6.8 GB of downloads)
FLASHVSR_COMMIT = "b548a2cc2c929d3d4676f2973abe27876ec4dee7"  # OpenImagingLab/FlashVSR_Plus, 16 Oct 2026
WEIGHTS_REVISION = "46b79520946ac67dec94d1566490690953c7debc"  # ruchangyao/FlashVSR_plus on Hugging Face, 14 Oct 2026
ON_KAGGLE = os.path.isdir("/kaggle")
REPO_DIR = "/kaggle/tmp/FlashVSR_Plus" if ON_KAGGLE else os.path.abspath("FlashVSR_Plus")
MODELS_DIR = "/kaggle/tmp/flashvsr_models" if ON_KAGGLE else os.path.abspath("flashvsr_models")

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"  # no second copy of the weights in the xet cache

print("=" * 70)
print("🎞️ FlashVSR+ Video Upscaler")
print("📺 Created by: AIQUEST Academy | YouTube & X: @AIQuestAcademy")
print("=" * 70)
ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available")
try:
    smi = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip().splitlines()
    for i, line in enumerate(smi):
        print(f"✅ GPU {i}: {line}")
    if len(smi) < 2:
        print("⚠️ One GPU only: it works, but at about half the speed. For full speed use Settings → Accelerator → GPU T4 x2")
except Exception:
    print("❌ No GPU found: Settings → Accelerator → GPU T4 x2")


def fmt_time(seconds):
    seconds = int(seconds)
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


def run_quiet(label, cmd):
    """Run a command with one live spinner line; show its output only if it fails."""
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    reader = threading.Thread(target=lambda: lines.extend(proc.stdout), daemon=True)
    reader.start()
    frames = "⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏"
    while proc.poll() is None:
        sys.stdout.write(f"\r  {frames[int(time.time() * 4) % 10]} {label} | {fmt_time(time.time() - t0)}".ljust(80))
        sys.stdout.flush()
        time.sleep(0.25)
    reader.join(timeout=5)
    sys.stdout.write("\r" + " " * 80 + "\r")
    if proc.returncode != 0:
        print(f"  ❌ {label} failed:\n" + "".join(lines[-40:]))
        raise SystemExit(f"❌ {label} failed (see above)")
    print(f"  ✅ {label} ({fmt_time(time.time() - t0)})")


# ---- 1. FlashVSR+ code, pinned to a tested commit ----
print("\n🧩 Code and dependencies")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "init", "-q", REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "add", "origin", "https://github.com/OpenImagingLab/FlashVSR_Plus.git"], check=True)
run_quiet(f"FlashVSR+ @ {FLASHVSR_COMMIT[:10]}", ["git", "-C", REPO_DIR, "fetch", "-q", "--depth", "1", "origin", FLASHVSR_COMMIT])
subprocess.run(["git", "-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD"], check=True)  # also undoes stale edits

# ---- 2. Dependencies (Kaggle's own torch / triton / opencv stay) ----
packages = ["gradio==5.29.0", "huggingface_hub[hf_xet]<1.0", "safetensors>=0.5", "einops>=0.8", "imageio-ffmpeg>=0.6"]
try:
    import cv2  # noqa: F401
except ImportError:
    packages.append("opencv-python-headless>=4.10")
run_quiet("Python packages", [sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "--timeout", "120",
                              "--retries", "5", *packages])
import torch
try:
    import triton
    triton_version = triton.__version__
except Exception as e:
    triton_version = f"not usable ({type(e).__name__}) - the engine falls back to PyTorch attention"
print(f"  ✅ PyTorch {torch.__version__} | Triton {triton_version}")

# ---- 3. Model weights (parallel; big files from an attached Kaggle Dataset are reused) ----
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
WEIGHTS_REPO = "ruchangyao/FlashVSR_plus"
modes = ["turbo", "base"] if GET_BASE_MODEL else ["turbo"]
FILES = [f"{m}/{f}" for m in modes for f in ("dit.safetensors", "lq_proj_in.safetensors")] + ["TCDecoder.ckpt"]
SIZES = {info.path: getattr(info, "size", 0) or 0
         for info in api.get_paths_info(WEIGHTS_REPO, FILES, revision=WEIGHTS_REVISION)}

attached = {}
if os.path.isdir("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            path = os.path.join(root, f)
            try:
                attached.setdefault((f, os.path.getsize(path)), []).append(path)
            except OSError:
                pass


def local_path(filename):
    return os.path.join(MODELS_DIR, filename)


def attached_copy(filename, size):
    """The same file in an attached Kaggle Dataset. turbo/ and base/ hold files with the same names and sizes, so
    those only count when they sit in a folder of the same name."""
    candidates = attached.get((os.path.basename(filename), size), [])
    folder = os.path.dirname(filename)
    if folder:
        candidates = [c for c in candidates if f"/{folder}/" in c]
    return candidates[0] if candidates else None


def fetch(filename):
    dest = local_path(filename)
    if os.path.exists(dest):
        return f"✓ {filename}"
    size = SIZES.get(filename, 0)
    source = attached_copy(filename, size) if size > 50e6 else None
    if source:
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        os.symlink(source, dest)
        return f"✓ {filename} (from attached dataset)"
    for attempt in range(1, 4):
        try:
            hf_hub_download(repo_id=WEIGHTS_REPO, filename=filename, revision=WEIGHTS_REVISION, local_dir=MODELS_DIR)
            return f"✅ {filename}" + (f" ({size / 1e9:.1f} GB)" if size > 50e6 else "")
        except Exception as e:
            if attempt == 3:
                raise
            print(f"  ⚠️ {filename}: {e} - retrying ({attempt}/3)")
            time.sleep(5 * attempt)


def folder_bytes(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            p = os.path.join(root, f)
            if not os.path.islink(p):
                try:
                    total += os.path.getsize(p)
                except OSError:
                    pass
    return total


os.makedirs(MODELS_DIR, exist_ok=True)
todo = [f for f in FILES if not os.path.exists(local_path(f))]
need = sum(SIZES.get(f, 0) for f in todo if not attached_copy(f, SIZES.get(f, 0)))
free_gb = shutil.disk_usage(MODELS_DIR).free / 1024**3
print(f"\n🚀 Downloading {need / 1e9:.1f} GB of weights to {MODELS_DIR} ({free_gb:.0f} GB free)...")
if need / 1024**3 > free_gb - 2:
    raise SystemExit(f"❌ Not enough disk space: need {need / 1e9:.0f} GB, {free_gb:.0f} GB free.")
done = threading.Event()
print_lock = threading.Lock()
bar_shown = [False]


def say(message):  # a normal line printed above the live bar
    with print_lock:
        if bar_shown[0]:
            sys.stdout.write("\r" + " " * 100 + "\r")
            bar_shown[0] = False
        print(message, flush=True)


def monitor():
    # One live line: GB done / total, %, smoothed speed, ETA. hf_xet writes into partial files, so bytes on disk
    # (minus what was already there) is the real progress.
    base = folder_bytes(MODELS_DIR)
    total = max(need, 1.0)
    start, samples = time.time(), []
    while not done.wait(1.0):
        got = folder_bytes(MODELS_DIR) - base
        now = time.time()
        samples = [(t, b) for t, b in samples if now - t <= 10] + [(now, got)]
        speed = (samples[-1][1] - samples[0][1]) / max(samples[-1][0] - samples[0][0], 1e-6) if len(samples) > 1 else 0
        got = min(got, total)
        frac = got / total
        fill = int(frac * 28)
        eta = fmt_time(max(0.0, total - got) / speed) if speed > 1e6 else "--"
        line = (f"  ⬇️ [{'█' * fill}{'░' * (28 - fill)}] {got / 1e9:5.1f} / {total / 1e9:.1f} GB {frac * 100:3.0f}% | "
                f"{speed / 1e6:4.0f} MB/s | ETA {eta} | {fmt_time(now - start)}")
        with print_lock:
            sys.stdout.write("\r" + line.ljust(100))
            sys.stdout.flush()
            bar_shown[0] = True


threading.Thread(target=monitor, daemon=True).start()
t0 = time.time()
try:
    with ThreadPoolExecutor(max_workers=5) as pool:
        jobs = [pool.submit(fetch, f) for f in sorted(FILES, key=lambda f: -SIZES.get(f, 0))]
        for job in jobs:
            result = job.result()
            if "GB" in result or "attached" in result:  # only the big files get a line
                say("  " + result)
finally:
    done.set()
    time.sleep(1.1)
say(f"  ✅ {need / 1e9:.1f} GB downloaded in {fmt_time(time.time() - t0)}"
    + (f" (avg {need / 1e6 / max(time.time() - t0, 1):.0f} MB/s)" if need else ""))
shutil.rmtree(os.path.join(MODELS_DIR, ".cache"), ignore_errors=True)

# Settings for Step 2's engine (saved to a file, so Step 2 can be re-run alone after a kernel restart)
SETTINGS = {"FLASHVSR_REPO": REPO_DIR, "FLASHVSR_MODELS": MODELS_DIR}
os.environ.update(SETTINGS)
with open("flashvsr_settings.json", "w") as f:
    json.dump(SETTINGS, f)
print("\n" + "=" * 70)
print(f"✅ Step 1 done in {fmt_time(time.time() - T_START)} | disk free: {shutil.disk_usage(MODELS_DIR).free / 1024**3:.0f} GB")
print("=" * 70)

---
## 🚀 Step 2: Start the engine + UI

Loads the model once and keeps it in VRAM, split across the two GPUs:
- **GPU 0:** input (LQ) projection + the first DiT blocks
- **GPU 1:** the remaining DiT blocks + output head + decoder

At startup each GPU tunes the sparse attention kernel (a few seconds), then a short warm-up compiles everything. The cell then prints the **Cloudflare** and **Gradio** links. Every video prints a short summary here: frames per second, seconds per frame, peak VRAM, and how long each GPU spends per chunk. The first video at a new size also moves the split point between the GPUs until both take the same time.

- The page shows a live progress bar; reloading the page or a dropped connection does not stop the job
- **Stop** ends the job after the current chunk. To restart the engine, re-run this cell
- The newest video is also at `<link>/aiq_video`

In [ ]:
# Step 2: FlashVSR+ engine (both T4s) + branded Gradio UI
# Writes run_flashvsr.py, then runs it: kernel tuning + model load (~2 min) → public links.
import codecs
import json
import os
import re
import subprocess
import sys
import time

RUN_SCRIPT_CODE = r'''"""FlashVSR+ video upscaler engine for Kaggle GPU T4 x2 - AIQUEST Academy.

Step 2 of the notebook writes this file as run_flashvsr.py and runs it. It drives the official FlashVSR+ code
(OpenImagingLab/FlashVSR_Plus, pinned in Step 1) with these changes for T4 GPUs:

* FP16 instead of BF16. T4 (Turing) has no BF16 tensor cores, so the released BF16 settings would run on a slow
  emulated path. The DiT is cast straight from its FP32 checkpoint to FP16; norms, softmax and the attention
  accumulators stay in FP32.
* Both GPUs work at once, on consecutive chunks. GPU 0 runs the input projection and the first DiT blocks of chunk n+1
  while GPU 1 runs the remaining blocks and the decoder for chunk n. Each block keeps its attention cache on the GPU
  that runs it, so the output is the same as on one GPU. The split point is measured on every new resolution and moved
  until both GPUs need the same time per chunk. Everything that crosses between the GPUs goes through pinned CPU
  memory: direct GPU-to-GPU copies hit illegal memory accesses on Kaggle's T4 pair.
* The sparse attention runs on FlashVSR+'s own Triton kernel, tuned at startup for the T4: the default 128x128 tiles
  need more shared memory than a T4 has. If the kernel cannot run, an exact PyTorch block-sparse path takes over.
* RoPE in FP32 with cached angles instead of FP64 (a T4 runs FP64 at 1/32 speed). This changes at most one FP16
  rounding step in a few elements per million.
* Streaming in and out: CPU threads resize the frames ahead of time, every chunk is decoded as soon as it is ready,
  converted to 8-bit on the GPU and piped to ffmpeg, so RAM stays flat on long videos.
* Any output size: each frame is mirror-padded to the model's multiple of 128 and the padding is cropped off the result,
  so 1080p comes out as 1920x1080 instead of a center-cropped 1920x1024.
"""
import copy
import gc
import glob
import json
import math
import os
import queue
import random
import re
import shutil
import subprocess
import sys
import threading
import time
import urllib.parse
import urllib.request
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

warnings.filterwarnings("ignore")

REPO_DIR = os.environ.get("FLASHVSR_REPO", "/kaggle/tmp/FlashVSR_Plus")
MODELS_DIR = os.environ.get("FLASHVSR_MODELS", "/kaggle/tmp/flashvsr_models")
WORK_DIR = os.environ.get("FLASHVSR_WORK_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd())
OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
SERVER_PORT = 7860
ENGINE_TEST = os.environ.get("FLASHVSR_ENGINE_TEST") == "1"  # import the functions only (CPU test harness)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ.setdefault("GRADIO_TEMP_DIR", os.path.join(OUTPUT_DIR, ".gradio_cache"))
for _p in (os.path.join(REPO_DIR, "operators", "flashvsr_attn"), REPO_DIR):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# expandable_segments maps GPU memory in a way that broke cross-GPU copies on Kaggle's T4 pair: keep the classic
# allocator (set before torch initializes CUDA).
if "expandable_segments" in os.environ.get("PYTORCH_CUDA_ALLOC_CONF", ""):
    os.environ.pop("PYTORCH_CUDA_ALLOC_CONF")

import cv2
import numpy as np
import psutil
import torch
import torch.nn.functional as F
from PIL import Image

# ==== FlashVSR+ code (from the pinned repo) ====
import models.attention as flash_attention
from inference.config import TEMPORAL_DOWNSAMPLE
from inference.decoders.tiny_model import build_tcdecoder
from inference.loading.loader import _load_context
from inference.loading.model_bundle import FlowMatchInferenceScheduler
from inference.postprocessing import WaveletColorCorrector
from inference.preprocessing import TemporalVideoStream
from inference.preprocessing.padding import target_frame_count
from inference.presets import model_paths_for_mode, stream_config_for_mode
from inference.runners.stream import _merge_segment_groups
from model_io import load_model_checkpoint
from models import BlockReplacementConfig, FlashVSRModelConfig, LQProjectionConfig, build_model
from models.attention import AttentionRuntime
from models.embeddings import rope_angles_3d, sinusoidal_embedding_1d
import importlib.util  # noqa: E402

# The kernel package imports Triton in its __init__; its layout helpers (plain PyTorch) are loaded on their own so the
# PyTorch attention path keeps working even without a usable Triton.
_spec = importlib.util.spec_from_file_location(
    "flashvsr_attn_layout", os.path.join(REPO_DIR, "operators", "flashvsr_attn", "flashvsr_attn", "layout.py"))
_layout = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_layout)
block_mask_to_packed, validate_stream_tensors = _layout.block_mask_to_packed, _layout.validate_stream_tensors

try:
    import triton  # noqa: F401
    from flashvsr_attn.triton_kernel import _stream_block_sparse_fwd
    TRITON_ERROR = None
except Exception as _e:  # no Triton (CPU test machine) or a broken install: the PyTorch path takes over
    _stream_block_sparse_fwd = None
    TRITON_ERROR = f"{type(_e).__name__}: {_e}"


def _gb(n):
    return f"{n / 1024**3:.1f} GB"


def _fmt_secs(seconds):
    seconds = int(round(seconds))
    if seconds >= 3600:
        return f"{seconds // 3600}h{seconds % 3600 // 60:02d}m"
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


class ConsoleBar:
    """One live progress line per phase, redrawn every second with a carriage return (Kaggle's cell output overwrites
    it in place). A finished phase is replaced by a one-line summary. Step-based phases interpolate inside the current
    step from the previous step's duration, so the bar keeps moving and the ETA counts down between steps."""

    WIDTH, LINE = 24, 104

    def __init__(self, stream):
        self.out = stream
        self.lock = threading.RLock()
        self.phase = None
        self.visible = False
        self.text_pending = False  # another writer is in the middle of a line: don't draw over it
        threading.Thread(target=self._loop, daemon=True).start()

    def start(self, key, label, total=None, unit=""):
        with self.lock:
            self.finish()
            now = time.time()
            self.phase = dict(key=key, label=label, total=total, unit=unit, done=0, t0=now, t_done=now, dt=None)
            self._draw()

    def update(self, done, total=None):
        with self.lock:
            p = self.phase
            if p is None:
                return
            now = time.time()
            if total:
                p["total"] = total
            if done > p["done"]:
                p["dt"] = (now - p["t_done"]) / (done - p["done"])
                p["t_done"], p["done"] = now, done
            self._draw()

    def finish(self, note=""):
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            elapsed = time.time() - p["t0"]
            line = f"  ✅ {p['label']}"
            if p["total"]:
                line += f" {p['done']}/{p['total']} {p['unit']}"
            line += f" in {_fmt_secs(elapsed) if elapsed >= 1 else f'{elapsed:.1f}s'}"
            if note:
                line += f" {note}"
            self._clear()
            self.out.write(line + "\n")
            self.out.flush()

    def abort(self, why="failed"):
        """End the live line of a phase that did not complete (no ✅ summary)."""
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            self._clear()
            self.out.write(f"  ❌ {p['label']} {why} after {_fmt_secs(time.time() - p['t0'])}\n")
            self.out.flush()

    def _render(self):
        p, now = self.phase, time.time()
        elapsed = now - p["t0"]
        if not p["total"]:
            spinner = "⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏"[int(now * 4) % 10]
            return f"  {spinner} {p['label']} | {_fmt_secs(elapsed)}"
        progress = p["done"]
        if p["dt"] and p["done"] < p["total"]:
            progress += min(0.97, (now - p["t_done"]) / p["dt"])
        frac = min(1.0, progress / p["total"])
        fill = int(round(frac * self.WIDTH))
        line = (f"  ⏳ {p['label']:<18} [{'█' * fill}{'░' * (self.WIDTH - fill)}] {frac * 100:3.0f}% | "
                f"{p['done']}/{p['total']} {p['unit']}")
        if p["dt"]:
            eta = max(0.0, (p["total"] - p["done"]) * p["dt"] - (now - p["t_done"]))
            line += f" | {p['dt']:.1f}s each | ETA {_fmt_secs(eta)}"
        return line + f" | {_fmt_secs(elapsed)}"

    def _draw(self):
        if self.phase is None or self.text_pending:
            return
        self.out.write("\r" + self._render().ljust(self.LINE))
        self.out.flush()
        self.visible = True

    def _clear(self):
        if self.visible:
            self.out.write("\r" + " " * self.LINE + "\r")
            self.visible = False

    def _loop(self):
        while True:
            time.sleep(1.0)
            with self.lock:
                self._draw()


class _BarAwareStream:
    """stdout / stderr wrapper: ordinary log lines clear the live bar first and print above it."""

    def __init__(self, stream, bar):
        self._stream, self._bar = stream, bar

    def write(self, data):
        with self._bar.lock:
            if data:
                self._bar._clear()
                self._bar.text_pending = not data.endswith("\n")
            return self._stream.write(data)

    def flush(self):
        self._stream.flush()

    def __getattr__(self, name):
        return getattr(self._stream, name)


BAR = ConsoleBar(sys.stdout)
if not ENGINE_TEST:
    sys.stdout = _BarAwareStream(sys.stdout, BAR)
    sys.stderr = _BarAwareStream(sys.stderr, BAR)

# ==== Devices ====
CUDA = torch.cuda.is_available()
if not CUDA and not ENGINE_TEST:
    raise SystemExit("❌ No GPU found. Settings → Accelerator → GPU T4 x2")
N_GPU = torch.cuda.device_count() if CUDA else 0
DEV_A = torch.device("cuda", 0) if CUDA else torch.device("cpu")                  # LQ projection + first DiT blocks
DEV_B = torch.device("cuda", 1) if N_GPU > 1 else DEV_A                          # last DiT blocks + head + decoder
DTYPE = torch.float16 if CUDA else torch.float32  # the CPU test harness may switch this before building models
NUM_BLOCKS = 30
if CUDA:
    torch.cuda.set_device(DEV_A)
    # FP16 GEMMs keep FP32 accumulation all the way (no FP16 split-K reductions): no overflow in the 8960-wide FFN.
    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
    torch.backends.cudnn.benchmark = True  # the decoder runs the same conv shapes for every frame
    if hasattr(torch.backends.cudnn, "benchmark_limit"):
        # Try only cuDNN's 3 best-ranked algorithms per new conv shape (default 10): every new output size used to
        # spend a long first chunk timing slow algorithms (FFT etc.) on the large decoder convolutions.
        torch.backends.cudnn.benchmark_limit = 3
    _props = [torch.cuda.get_device_properties(i) for i in range(N_GPU)]


# ==== T4 speed patch 1: RoPE in FP32 with cached angles ====
# The official apply_rope_3d rotates Q and K in FP64 and recomputes the angle table for every layer. The angles are
# identical here (same FP32 positions, cos/sin in FP64, built on the CPU), cached once per chunk; only the rotation
# runs in FP32 on the GPU.
_ROPE_CACHE = {}


def fast_apply_rope_3d(x, shape, *, temporal_offset=0):
    key = (dkey(x.device), tuple(shape), x.shape[-1], int(temporal_offset))
    table = _ROPE_CACHE.get(key)
    if table is None:
        if len(_ROPE_CACHE) > 32:
            _ROPE_CACHE.clear()
        angles = rope_angles_3d(shape, x.shape[-1], torch.device("cpu"), temporal_offset=temporal_offset)
        angles = angles.to(torch.float64)
        table = tuple(t.float().view(1, 1, angles.shape[0], -1).to(x.device) for t in (angles.cos(), angles.sin()))
        _ROPE_CACHE[key] = table
    cos, sin = table
    pairs = x.float().reshape(*x.shape[:-1], -1, 2)
    first, second = pairs[..., 0], pairs[..., 1]
    rotated = torch.stack((first * cos - second * sin, second * cos + first * sin), dim=-1)
    return rotated.flatten(-2).to(dtype=x.dtype)


flash_attention.apply_rope_3d = fast_apply_rope_3d


# ==== T4 speed patch 2: block-sparse stream attention (tuned Triton kernel, or an exact PyTorch path) ====
def triton_sparse_attention(q_windows, k_windows, v_windows, block_mask, config):
    """FlashVSR+'s Triton kernel (operators/flashvsr_attn) with selectable tiles; the official wrapper fixes the key
    tile at 64/128 and two pipeline stages, which do not fit the T4's 64 KB of shared memory at head_dim 128."""
    block_m, block_n, num_warps, num_stages = config
    batch, heads, query_blocks, key_blocks, head_dim = validate_stream_tensors(q_windows, k_windows, v_windows,
                                                                             block_mask)
    q = q_windows.contiguous().reshape(batch * heads, query_blocks, 128, head_dim)
    k = k_windows.contiguous().reshape(batch * heads, key_blocks, 128, head_dim)
    v = v_windows.contiguous().reshape(batch * heads, key_blocks, 128, head_dim)
    row_counts, col_idx = block_mask_to_packed(block_mask.to(device=q.device), ensure_nonempty=False)
    out = torch.empty_like(q)
    grid = (batch * heads * query_blocks * (128 // block_m),)
    _stream_block_sparse_fwd[grid](
        q, k, v, row_counts, col_idx, out,
        *q.stride(), *k.stride(), *v.stride(), *out.stride(),
        query_blocks, key_blocks, query_blocks, key_blocks, 1.0 / math.sqrt(head_dim),
        HEAD_DIM=head_dim, BLOCK_M=block_m, BLOCK_N=block_n, num_warps=num_warps, num_stages=num_stages)
    return out.reshape(batch, heads, query_blocks, 128, head_dim)


def _bmm_fp32(a, b):
    """a @ b with FP32 output (FP16 tensor-core inputs, FP32 accumulation, no FP16 rounding of the result)."""
    if a.is_cuda and _BMM_OUT_DTYPE[0]:
        try:
            return torch.bmm(a, b, out_dtype=torch.float32)
        except (TypeError, RuntimeError):
            _BMM_OUT_DTYPE[0] = False
    return torch.bmm(a, b).float() if a.is_cuda else torch.bmm(a.float(), b.float())


_BMM_OUT_DTYPE = [True]


def torch_sparse_attention(q_windows, k_windows, v_windows, block_mask, method="sdpa", budget_bytes=768 * 1024**2):
    """Exact block-sparse attention in PyTorch: each 128-token query block attends to its selected 128-token key
    blocks. Query blocks are sorted by how many key blocks they read and processed in groups, each group gathering its
    key blocks padded only to the group's own longest list.
    'sdpa': PyTorch's fused attention with a padding mask. 'sdpa_nomask': the same without a mask: padded key slots
    get one extra feature column that scores them at -1e4 (they vanish in the softmax), which lets the fused kernel
    run mask-free. 'matmul': cuBLAS GEMMs with an FP32 softmax in between."""
    batch, heads, query_blocks, _, head_dim = q_windows.shape
    key_blocks = k_windows.shape[2]
    bh = batch * heads
    row_counts, col_idx = block_mask_to_packed(block_mask.to(device=q_windows.device), ensure_nonempty=False)
    col_idx = col_idx.long()  # [BH, QB, KB]; slots past a row's count hold the sentinel index key_blocks
    q = q_windows.reshape(bh, query_blocks, 128, head_dim)
    extra = 8 if method == "sdpa_nomask" else 0  # 1 scoring column, padded to a multiple of 8 for the fused kernel
    k = q_windows.new_zeros(bh, key_blocks + 1, 128, head_dim + extra)
    v = q_windows.new_zeros(bh, key_blocks + 1, 128, head_dim + extra)
    k[:, :key_blocks, :, :head_dim] = k_windows.reshape(bh, key_blocks, 128, head_dim)
    v[:, :key_blocks, :, :head_dim] = v_windows.reshape(bh, key_blocks, 128, head_dim)
    if extra:
        q = F.pad(q, (0, extra))
        q[..., head_dim] = 1.0
        k[:, key_blocks, :, head_dim] = -1e4  # sentinel block: q.k = -1e4, exp(-1e4 / sqrt(128)) = 0
    widest = row_counts.max(dim=0).values  # [QB] longest key list of each query block over the heads
    order = torch.argsort(widest)
    widest_sorted = widest[order].tolist()
    order_list = order.tolist()
    rows = torch.arange(bh, device=q.device).view(bh, 1, 1)
    scale = 1.0 / math.sqrt(head_dim)
    out = torch.empty(bh, query_blocks, 128, head_dim, dtype=q_windows.dtype, device=q_windows.device)
    start = 0
    while start < query_blocks:
        # grow the group while its padded K/V (+ scores for matmul) fit the budget
        stop, longest = start, 0
        while stop < query_blocks:
            m = max(longest, int(widest_sorted[stop]), 1)
            per_block = bh * m * 128 * (head_dim + extra) * q.element_size() * 2
            if method == "matmul":
                per_block += bh * 128 * m * 128 * 4 * 2
            if stop > start and per_block * (stop - start + 1) > budget_bytes:
                break
            longest, stop = m, stop + 1
        sel = torch.tensor(order_list[start:stop], device=q.device)
        n = stop - start
        idx = col_idx[:, sel, :longest]
        kg = k[rows, idx].reshape(bh * n, longest * 128, head_dim + extra)
        vg = v[rows, idx].reshape(bh * n, longest * 128, head_dim + extra)
        qg = q[:, sel].reshape(bh * n, 128, head_dim + extra)
        if method == "sdpa_nomask":
            res = F.scaled_dot_product_attention(qg[:, None], kg[:, None], vg[:, None], scale=scale)
            res = res.squeeze(1)[..., :head_dim]
        else:
            slot = torch.arange(longest, device=q.device).view(1, 1, longest)
            valid = (slot < row_counts[:, sel, None]).repeat_interleave(128, dim=-1).reshape(bh * n, 1, longest * 128)
            if method == "sdpa":
                res = F.scaled_dot_product_attention(qg[:, None], kg[:, None], vg[:, None],
                                                     attn_mask=valid[:, None]).squeeze(1)
            else:
                scores = _bmm_fp32(qg, kg.transpose(1, 2)).mul_(scale)
                probs = scores.masked_fill_(~valid, float("-inf")).softmax(dim=-1)
                del scores
                res = torch.bmm(probs.to(vg.dtype), vg)
                del probs
        out[:, sel] = res.reshape(bh, n, 128, head_dim).to(out.dtype)
        del kg, vg, qg, res
        start = stop
    return out.reshape(batch, heads, query_blocks, 128, head_dim)


def _efficient_varlen(q, k, v, cu_q, cu_k, max_q, max_k, scale):
    """PyTorch's fused (memory-efficient) attention kernel on packed variable-length sequences:
    q [1, Mq, 1, D] and k/v [1, Mk, 1, D] hold many independent sequences delimited by cu_q / cu_k."""
    if q.is_cuda:
        return torch.ops.aten._efficient_attention_forward(
            q, k, v, None, cu_q, cu_k, max_q, max_k, 0.0, 0, False, scale=scale)[0]
    out = torch.empty_like(q)  # CPU (test machine): the same computation, one sequence at a time
    for i in range(cu_q.numel() - 1):
        qs, qe, ks, ke = int(cu_q[i]), int(cu_q[i + 1]), int(cu_k[i]), int(cu_k[i + 1])
        out[0, qs:qe, 0] = F.scaled_dot_product_attention(q[0, qs:qe, 0][None], k[0, ks:ke, 0][None],
                                                          v[0, ks:ke, 0][None], scale=scale)[0]
    return out


def varlen_sparse_attention(q_windows, k_windows, v_windows, block_mask, budget_bytes=1024 * 1024**2):
    """Exact block-sparse attention with no padding and no mask: every (head, query block) is one sequence whose keys
    are exactly its selected key blocks, packed back to back and run through the fused kernel's variable-length mode.
    Query blocks are processed in groups that fit the memory budget."""
    batch, heads, query_blocks, _, head_dim = q_windows.shape
    key_blocks = k_windows.shape[2]
    bh = batch * heads
    mask = block_mask.to(device=q_windows.device).reshape(bh, query_blocks, key_blocks).transpose(0, 1)  # [QB, BH, KB]
    counts = mask.sum(dim=-1, dtype=torch.int32)                                    # keys per (query block, head)
    pairs = mask.nonzero()                     # [P, 3] (query block, head, key block), ordered by sequence
    seq_end = torch.cumsum(counts.flatten(), 0)                                      # pairs up to each sequence
    per_qb = counts.sum(dim=1).tolist()                                              # pairs per query block
    widest = counts.max(dim=1).values.tolist()
    seq_end_list = seq_end.tolist()
    q = q_windows.reshape(bh, query_blocks, 128, head_dim).transpose(0, 1).contiguous()  # [QB, BH, 128, D]
    k = k_windows.reshape(bh, key_blocks, 128, head_dim)
    v = v_windows.reshape(bh, key_blocks, 128, head_dim)
    out = torch.empty_like(q)
    scale = 1.0 / math.sqrt(head_dim)
    pair_bytes = 128 * head_dim * q.element_size() * 2
    start = 0
    while start < query_blocks:
        stop, used = start, 0
        while stop < query_blocks and (stop == start or used + per_qb[stop] * pair_bytes <= budget_bytes):
            used += per_qb[stop] * pair_bytes
            stop += 1
        first = seq_end_list[start * bh - 1] if start else 0
        last = seq_end_list[stop * bh - 1]
        sel = pairs[first:last]
        kg = k[sel[:, 1], sel[:, 2]].reshape(1, -1, 1, head_dim)
        vg = v[sel[:, 1], sel[:, 2]].reshape(1, -1, 1, head_dim)
        n_seq = (stop - start) * bh
        cu_q = torch.arange(0, (n_seq + 1) * 128, 128, device=q.device, dtype=torch.int32)
        cu_k = torch.zeros(n_seq + 1, device=q.device, dtype=torch.int32)
        cu_k[1:] = (seq_end[start * bh:stop * bh] - first) * 128
        res = _efficient_varlen(q[start:stop].reshape(1, -1, 1, head_dim), kg, vg, cu_q, cu_k, 128,
                                int(max(widest[start:stop])) * 128, scale)
        out[start:stop] = res.reshape(stop - start, bh, 128, head_dim)
        del kg, vg, res, sel
        start = stop
    return out.transpose(0, 1).reshape(batch, heads, query_blocks, 128, head_dim)


ATTN = {}  # device key -> ("triton", (block_m, block_n, warps, stages)) or ("torch", "sdpa" | "sdpa_nomask" | "matmul")


def dkey(device):
    device = torch.device(device)
    return f"{device.type}:{device.index or 0}"


def xfer(tensor, device):
    """Copy to another device; GPU to GPU goes through CPU memory (no peer-to-peer copies, see the docstring)."""
    device = torch.device(device)
    if tensor.device.type == "cuda" and device.type == "cuda" and tensor.device != device:
        return tensor.to("cpu").to(device)
    return tensor.to(device)


def move_module(module, device):
    device = torch.device(device)
    current = next(module.parameters(), None)
    if current is not None and current.device.type == "cuda" and device.type == "cuda" and current.device != device:
        module.to("cpu")
    return module.to(device)


def sparse_attention(q_windows, k_windows, v_windows, block_mask, *, ensure_nonempty=False, block_m=128):
    # The layout builder already gives every query row at least one key block (ensure_nonempty is always False).
    kind, config = ATTN.get(dkey(q_windows.device), ("torch", "sdpa"))
    if kind == "triton":
        return triton_sparse_attention(q_windows, k_windows, v_windows, block_mask, config)
    if config == "varlen":
        return varlen_sparse_attention(q_windows, k_windows, v_windows, block_mask)
    return torch_sparse_attention(q_windows, k_windows, v_windows, block_mask, method=config)


flash_attention.stream_block_sparse_attention = sparse_attention

# (block_m, block_n, warps, stages). A T4 has 64 KB of shared memory per block; FP16 Q/K/V tiles at head_dim 128 take
# block x 256 bytes each, so 64 x 32 and 32 x 64 fit with room to spare.
TRITON_CONFIGS = [(64, 32, 4, 1), (64, 64, 4, 1), (32, 64, 4, 1), (32, 32, 4, 1), (64, 16, 4, 1), (64, 32, 4, 2),
                  (128, 32, 8, 1), (64, 64, 8, 1)]


def _attn_label(kind, config):
    if kind == "triton":
        bm, bn, warps, stages = config
        return f"Triton {bm}x{bn} tiles, {warps} warps, {stages} stage{'s' if stages > 1 else ''}"
    return {"sdpa": "PyTorch fused attention + mask", "sdpa_nomask": "PyTorch fused attention, mask-free",
            "matmul": "PyTorch cuBLAS matmul + softmax",
            "varlen": "PyTorch fused attention, exact lengths"}.get(config, f"PyTorch {config}")


def _short_error(e):
    text = str(e)
    match = re.search(r"Required: (\d+), Hardware limit: (\d+)", text)
    if match:
        return f"needs {int(match.group(1)) // 1024} KB shared memory, T4 has {int(match.group(2)) // 1024} KB"
    return f"{type(e).__name__}: {text.strip().splitlines()[0][:140] if text.strip() else ''}"


def tune_attention(device, candidates):
    """Run each candidate on a stream-sized problem (12 heads, 96 query x 240 key blocks, 25% density), check it
    against an FP32 reference and time it. Keeps the fastest correct one for `device`. Returns
    [(kind, config, ms or None, note)]."""
    key = dkey(device)
    if device.type != "cuda":
        ATTN[key] = ("torch", "sdpa")
        return [("torch", "sdpa", None, "CPU")]
    rows = []
    with torch.cuda.device(device):
        g = torch.Generator(device=device).manual_seed(0)
        q = torch.randn((1, 12, 96, 128, 128), generator=g, device=device, dtype=DTYPE)
        k = torch.randn((1, 12, 240, 128, 128), generator=g, device=device, dtype=DTYPE)
        v = torch.randn((1, 12, 240, 128, 128), generator=g, device=device, dtype=DTYPE)
        mask = torch.rand((1, 12, 96, 240), generator=g, device=device) < 0.25
        mask[..., 0] = True
        reference = torch_sparse_attention(q.float(), k.float(), v.float(), mask, method="matmul")
        torch.cuda.synchronize(device)

        def run(kind, config):
            if kind == "triton":
                return triton_sparse_attention(q, k, v, mask, config)
            if config == "varlen":
                return varlen_sparse_attention(q, k, v, mask)
            return torch_sparse_attention(q, k, v, mask, method=config)

        def bench(kind, config, reps=3):
            run(kind, config)
            torch.cuda.synchronize(device)
            start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            start.record()
            for _ in range(reps):
                run(kind, config)
            end.record()
            end.synchronize()
            return start.elapsed_time(end) / reps

        for kind, config in candidates:
            if kind == "triton" and _stream_block_sparse_fwd is None:
                rows.append((kind, config, None, f"Triton unavailable ({TRITON_ERROR})"))
                continue
            try:
                out = run(kind, config)
                torch.cuda.synchronize(device)
                err = (out.float() - reference).abs().max().item()
                if not math.isfinite(err) or err > 2e-2:
                    rows.append((kind, config, None, f"wrong results (max error {err:.3g})"))
                    continue
                rows.append((kind, config, bench(kind, config), f"max error {err:.1e}"))
            except Exception as e:  # out of shared memory, compiler error...
                rows.append((kind, config, None, _short_error(e)))
                torch.cuda.synchronize(device)
    good = [r for r in rows if r[2] is not None]
    best = min(good, key=lambda r: r[2]) if good else ("torch", "sdpa", None, "")
    ATTN[key] = (best[0], best[1])
    return rows


# ==== Models ====
def make_config(mode):
    """The official preset for this mode, switched to FP16 and the Triton stream kernel (the kernel itself is chosen
    per GPU by tune_attention)."""
    cfg = stream_config_for_mode(mode, device="cpu", hardware_profile="generic")
    cfg.attn_kernel = "triton_block_sparse"
    cfg.dtype = "fp16"
    cfg.validate()
    return cfg


def model_config_for(cfg):
    return FlashVSRModelConfig(
        attention=cfg.attention_config(),
        lq_projection=LQProjectionConfig(enabled=True, num_injection_layers=cfg.lq_projection_layers),
        replacement=BlockReplacementConfig(layers=cfg.replacement_layers))


def available_modes():
    return [m for m in ("turbo", "base")
            if all(os.path.exists(os.path.join(MODELS_DIR, m, f)) for f in ("dit.safetensors", "lq_proj_in.safetensors"))]


def build_master(mode):
    """DiT + LQ projection as the official loader builds them (FP32 init, checkpoint load, 95% coverage check), then
    cast once to FP16. Kept in CPU RAM so that switching modes or moving blocks never re-reads the disk."""
    cfg = make_config(mode)
    paths = model_paths_for_mode(Path(REPO_DIR), mode, os.path.join(MODELS_DIR, mode))
    model = build_model(model_config_for(cfg))
    dit_report = load_model_checkpoint(model, paths.dit, min_coverage=0.95)
    lq_report = load_model_checkpoint(model.LQ_proj_in, paths.lq_projection, component_prefix="LQ_proj_in.",
                                      min_coverage=0.95)
    missing = [k for k in dit_report.missing if not k.startswith("LQ_proj_in.")]
    missing += [f"LQ_proj_in.{k}" for k in lq_report.missing]
    model = model.to(DTYPE).eval().requires_grad_(False)
    gc.collect()
    return model, cfg, missing


def build_decoder(cfg):
    module = build_tcdecoder(new_channels=list(cfg.decoder.tiny_channels), device="cpu", dtype=DTYPE,
                             new_latent_channels=cfg.decoder.tiny_latent_channels)
    state = torch.load(os.path.join(MODELS_DIR, "TCDecoder.ckpt"), map_location="cpu", weights_only=True)
    if isinstance(state, dict) and "decoder" in state:
        state = state["decoder"]
    module.load_state_dict(state, strict=False)  # as the official loader
    module = module.to(DEV_B).eval().requires_grad_(False)
    if DEV_B.type == "cuda" and cfg.decoder.tiny_decoder_channels_last:
        module.to(memory_format=torch.channels_last)
        module.use_channels_last = True
    module.clean_mem()
    return module


class SplitDiT:
    """One GPU copy of the DiT, cut between blocks: blocks [0, k) with the input layers on DEV_A, blocks [k, 30) and
    the output head on DEV_B."""

    def __init__(self, master, mode, cfg, k):
        self.mode, self.cfg = mode, cfg
        self.model = copy.deepcopy(master)
        m = self.model
        for part in (m.patch_embedding, m.text_embedding, m.time_embedding, m.time_projection, m.LQ_proj_in):
            part.to(DEV_A)
        m.head.to(DEV_B)
        self.k = max(1, min(NUM_BLOCKS, int(k)))
        self.where = []
        for j, block in enumerate(m.blocks):
            device = DEV_A if j < self.k else DEV_B
            block.to(device)
            self.where.append(device)

    def move_split(self, new_k, caches):
        """Move blocks (and their attention caches) across the GPUs so that blocks [0, new_k) live on DEV_A."""
        new_k = max(1, min(NUM_BLOCKS, int(new_k)))
        for j in range(NUM_BLOCKS):
            target = DEV_A if j < new_k else DEV_B
            if self.where[j] != target:
                move_module(self.model.blocks[j], target)
                if caches[j] is not None:
                    caches[j] = tuple(xfer(t, target) for t in caches[j])
                self.where[j] = target
        self.k = new_k
        if CUDA:
            for d in {DEV_A, DEV_B}:
                torch.cuda.synchronize(d)


# ==== Video input / output ====
def probe_video(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise ValueError(f"Could not open the video: {path}")
    info = dict(width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                fps=float(cap.get(cv2.CAP_PROP_FPS) or 0.0), frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0))
    cap.release()
    if info["fps"] <= 0 or not math.isfinite(info["fps"]):
        info["fps"] = 30.0
    return info


def read_frames(path, max_frames=None):
    """All frames (or the first max_frames) as RGB uint8 arrays at the source size."""
    cap = cv2.VideoCapture(str(path))
    frames = []
    try:
        while max_frames is None or len(frames) < max_frames:
            ok, frame = cap.read()
            if not ok:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    finally:
        cap.release()
    if not frames:
        raise ValueError("No frames could be decoded from the video.")
    return frames


def output_size(width, height, choice):
    """Output (w, h), both even, for a size choice such as '4x' or '1080p'."""
    choice = str(choice)
    match = re.match(r"\s*(\d+(?:\.\d+)?)\s*[x×]", choice)
    if match:
        scale = float(match.group(1))
        return max(128, int(round(width * scale / 2)) * 2), max(128, int(round(height * scale / 2)) * 2)
    short = int(re.search(r"(\d+)\s*p", choice).group(1))
    long_side = max(width, height) * short / min(width, height)
    widescreen = int(round(short * 16 / 9 / 2)) * 2  # 854x480 is 16:9 rounded: give 1920x1080, not 1922x1080
    long_side = widescreen if abs(long_side - widescreen) <= 0.01 * widescreen else int(round(long_side / 2)) * 2
    return (long_side, short) if width >= height else (short, long_side)


def model_size(w, h):
    return int(math.ceil(h / 128) * 128), int(math.ceil(w / 128) * 128)  # (Hm, Wm)


def prepare_frame(frame, out_w, out_h, model_h, model_w):
    """Bicubic resize to the output size (PIL, like the official streaming path), then mirror-pad to the model size."""
    if frame.shape[1] != out_w or frame.shape[0] != out_h:
        frame = np.asarray(Image.fromarray(frame).resize((out_w, out_h), resample=Image.Resampling.BICUBIC))
    pad_h, pad_w = model_h - out_h, model_w - out_w
    if pad_h or pad_w:
        frame = np.pad(frame, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
    return np.ascontiguousarray(frame)


def lq_chunks(frames, out_w, out_h, model_h, model_w, workers=3, lookahead=48, tap=None):
    """[1, 3, F, Hm, Wm] uint8 tensors: 17 frames, then 16 at a time. CPU threads prepare the frames ahead. They stay
    8-bit through the frame buffer (4x less copying than float32) and become [-1, 1] floats on the GPU (to_unit).
    tap(frame) receives every prepared frame cropped to the output size: the bicubic 'before' of the comparison."""
    with ThreadPoolExecutor(max_workers=workers) as pool:
        pending, index, batch, size = [], 0, [], 17
        while index < len(frames) or pending:
            while index < len(frames) and len(pending) < lookahead:
                pending.append(pool.submit(prepare_frame, frames[index], out_w, out_h, model_h, model_w))
                index += 1
            batch.append(pending.pop(0).result())
            if tap is not None:
                tap(batch[-1][:out_h, :out_w])
            if len(batch) == size or (index >= len(frames) and not pending):
                array = np.stack(batch)
                batch, size = [], 16
                yield torch.from_numpy(array).permute(3, 0, 1, 2).unsqueeze(0)


def to_unit(frames, device):
    """uint8 frames -> [-1, 1] float32 on the device: the official x / 127.5 - 1 in FP32, so the values are identical."""
    return frames.to(device).float().div_(127.5).sub_(1.0)


def ffmpeg_exe():
    found = shutil.which("ffmpeg")
    if found:
        return found
    import imageio_ffmpeg
    return imageio_ffmpeg.get_ffmpeg_exe()


class VideoWriter:
    """Raw RGB frames piped to ffmpeg (libx264) from a background thread."""

    def __init__(self, path, width, height, fps, crf=17, preset="veryfast", queue_size=4):
        self.path, self.frames = path, 0
        cmd = [ffmpeg_exe(), "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24",
               "-s", f"{width}x{height}", "-r", f"{fps:.6f}", "-i", "-", "-c:v", "libx264", "-preset", preset,
               "-crf", str(int(crf)), "-pix_fmt", "yuv420p", "-movflags", "+faststart", path]
        self.proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE)
        self.queue = queue.Queue(maxsize=queue_size)
        self.error = None
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()

    def _run(self):
        while True:
            item = self.queue.get()
            if item is None:
                break
            if self.error is not None:
                continue
            try:
                self.proc.stdin.write(item.tobytes())
                self.frames += item.shape[0]
            except Exception as e:
                self.error = e

    def push(self, frames_uint8):
        if self.error is not None:
            raise RuntimeError(f"ffmpeg stopped: {self.error}")
        self.queue.put(frames_uint8)

    def close(self):
        self.queue.put(None)
        self.thread.join()
        try:
            self.proc.stdin.close()
        except Exception:
            pass
        rc = self.proc.wait()
        err = self.proc.stderr.read().decode(errors="replace").strip()
        if rc != 0 or self.error is not None:
            raise RuntimeError(f"ffmpeg failed ({rc}): {err or self.error}")

    def abort(self):
        try:
            self.proc.kill()
        except Exception:
            pass


def mux_audio(video_path, source_path, destination, seconds):
    """Copy the source's audio track (re-encoded to AAC) next to the new video; video-only if there is none. The audio
    is cut to the video's length (-shortest would drop the last frame when the audio ends a few ms early)."""
    cmd = [ffmpeg_exe(), "-y", "-loglevel", "error", "-i", video_path, "-t", f"{seconds:.3f}", "-i", source_path,
           "-map", "0:v:0", "-map", "1:a:0?", "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
           "-movflags", "+faststart", destination]
    done = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if done.returncode != 0:
        shutil.move(video_path, destination)
        return False
    os.remove(video_path)
    return True


# ==== The two-GPU stream pipeline ====
class Cancelled(Exception):
    pass


CANCEL = threading.Event()
SPLIT_CACHE = {}  # (mode, Hm, Wm, temporal compression) -> measured best split point


class _Clock:
    """GPU time between marks (CUDA events), or wall time on the CPU."""

    def __init__(self, device):
        self.cuda = device.type == "cuda"
        self.marks = []

    def mark(self, name):
        if self.cuda:
            event = torch.cuda.Event(enable_timing=True)
            event.record()
            self.marks.append((name, event))
        else:
            self.marks.append((name, time.perf_counter()))

    def spans(self):
        """{name: ms} for each mark, measured from the previous mark."""
        if not self.marks:
            return {}
        if self.cuda:
            self.marks[-1][1].synchronize()
        out = {}
        for (_, a), (name, b) in zip(self.marks, self.marks[1:]):
            ms = a.elapsed_time(b) if self.cuda else (b - a) * 1000.0
            out[name] = out.get(name, 0.0) + ms
        return out


def _project_lq(projection, video, index, frame_offset):
    """The official StreamRunner._project_lq_chunk: one latent frame of LQ tokens per 4 input frames (plus the
    warm-up frame at the very start), concatenated along the token axis."""
    conditions = None

    def append(current):
        nonlocal conditions
        if current is None:
            return
        conditions = list(current) if conditions is None else [torch.cat([a, b], dim=1)
                                                                for a, b in zip(conditions, current)]

    if index == 0:
        append(projection.stream_forward(video[:, :, :1]))
    for inner in range(4):
        start = 1 + (index * 4 + inner) * TEMPORAL_DOWNSAMPLE - frame_offset
        append(projection.stream_forward(video[:, :, start:start + TEMPORAL_DOWNSAMPLE]))
    return conditions


def best_split(fixed_a, fixed_b, block_ms):
    """Split point k minimizing the slower stage: A = fixed_a + blocks[:k], B = fixed_b + blocks[k:]."""
    best_k, best_cost = 1, float("inf")
    for k in range(1, NUM_BLOCKS + 1):
        cost = max(fixed_a + sum(block_ms[:k]), fixed_b + sum(block_ms[k:]))
        if cost < best_cost:
            best_k, best_cost = k, cost
    return best_k, best_cost


def run_pipeline(split, decoder, context, chunk_iter, *, n_source_frames, model_hw, out_hw, temporal_compression,
                 seed, color_fix, sink, on_chunk=None, n_chunks=None, rebalance=True):
    """Upscale one stream. GPU 0 (stage A) and GPU 1 (stage B) work on consecutive chunks at the same time.

    sink(video, index) receives each decoded chunk on DEV_B as [1, 3, F, out_h, out_w] in [-1, 1], already merged
    back onto the source timeline and cut to n_source_frames. Returns timing statistics."""
    cfg, m = split.cfg, split.model
    model_h, model_w = model_hw
    out_h, out_w = out_hw
    two_gpus = DEV_A != DEV_B
    stream = TemporalVideoStream(chunk_iter, temporal_compression, TEMPORAL_DOWNSAMPLE)
    factor = TEMPORAL_DOWNSAMPLE // temporal_compression
    spatial_windows = (model_h // 16 // 8) * (model_w // 16 // 8)
    topk = cfg.sparse_topk if cfg.sparse_topk is not None else int(
        spatial_windows * spatial_windows * cfg.sparse_topk_ratio) - 1
    cache_blocks, block_size = int(cfg.sparse_kv_ratio), cfg.block_size
    corrector = WaveletColorCorrector(levels=cfg.color_correction.levels,
                                      chunk_size=cfg.color_correction.chunk_size) if color_fix else None
    area = model_h * model_w
    parallel_decode = [area <= 1_000_000]  # the official 'auto' rule, checked against free VRAM below
    if parallel_decode[0] and DEV_B.type == "cuda":
        free_mib = torch.cuda.mem_get_info(DEV_B)[0] / 1024**2
        parallel_decode[0] = free_mib >= 7680.0 * area / (1536 * 2560) + 768.0

    sched_a, sched_b = FlowMatchInferenceScheduler(cfg.sigma_shift), FlowMatchInferenceScheduler(cfg.sigma_shift)
    timestep_a = sched_a.timesteps(1, DEV_A, DTYPE)[0]
    timestep_b = sched_b.timesteps(1, DEV_B, DTYPE)[0]
    caches = [None] * NUM_BLOCKS
    state = {"error": None, "sent": -1, "done": -1, "produced": 0, "pending_k": None, "moves": 0,
             "chunks": [], "first_chunk_at": None}
    cond = threading.Condition()
    handoff = queue.Queue(maxsize=1)
    staged = two_gpus and DEV_A.type == "cuda"  # chunk hand-off through pinned CPU buffers
    split_key = (split.mode, model_h, model_w, temporal_compression)

    with torch.inference_mode():
        context = (context.unsqueeze(0) if context.ndim == 2 else context).to(DEV_A, DTYPE)
        timestep = timestep_a.expand(1).cpu()  # the FP64 sinusoid on the CPU, then to the GPU (as the official)
        t_embed = m.time_embedding(sinusoidal_embedding_1d(m.freq_dim, timestep).to(DEV_A, DTYPE))
        t_mod = m.time_projection(t_embed).unflatten(1, (6, m.dim))
        ctx = m.text_embedding(context)
        per_device = {dkey(DEV_A): (t_mod, ctx), dkey(DEV_B): (xfer(t_mod, DEV_B), xfer(ctx, DEV_B))}
        t_embed_b = xfer(t_embed, DEV_B)
    generator = torch.Generator(device=DEV_A).manual_seed(int(seed))
    m.LQ_proj_in.clear_cache()
    decoder.clean_mem()

    def run_blocks(x, start, stop, index, token_shape, write_cache, conditions, clock):
        for j in range(start, stop):
            if conditions is not None and j < len(conditions):
                x = x + conditions[j].to(device=x.device, dtype=x.dtype)
            t_mod_d, ctx_d = per_device[dkey(x.device)]
            runtime = AttentionRuntime(
                token_shape=token_shape, temporal_offset=4 * index, stream_cache=caches[j], stream_topk=topk,
                stream_cache_blocks=cache_blocks, stream_block_size=block_size,
                stream_ffn_chunk_tokens=cfg.stream_ffn_chunk_tokens, stream_block_m=cfg.stream_block_m,
                stream_compact_cache=False, stream_write_cache=write_cache, stream_release_workspaces=True)
            caches[j] = None
            x, caches[j] = m.blocks[j].forward_stream(x, ctx_d, t_mod_d, runtime, None)
            del runtime
            clock.mark(f"block{j}")
        return x

    def fail(error):
        with cond:
            if state["error"] is None:
                state["error"] = error
            cond.notify_all()

    # ---- stage B: last blocks, head, scheduler step, decoder, output (GPU 1) ----
    def stage_b():
        try:
            if CUDA:
                torch.cuda.set_device(DEV_B)
            with torch.inference_mode():
                while True:
                    msg = handoff.get()
                    if msg is None or state["error"] is not None:
                        return
                    index = msg["index"]
                    clock = _Clock(DEV_B)
                    if staged:
                        msg["ready"].synchronize()  # GPU 0 finished writing the pinned buffers
                    x_in = msg["x"].to(DEV_B, non_blocking=staged)
                    noise_b = msg["noise"].to(DEV_B, non_blocking=staged)
                    lq = to_unit(msg["lq"], DEV_B)  # first: a pageable copy waits for the GPU's queued work
                    clock.mark("start")
                    x = run_blocks(x_in, msg["k"], NUM_BLOCKS, index, msg["token_shape"], not msg["last"],
                                   None, clock)
                    del x_in
                    prediction = m.unpatchify(m.head(x, t_embed_b), msg["token_shape"])
                    latents = sched_b.step(prediction, timestep_b, noise_b)
                    del x, prediction
                    if index == 0 and not bool(torch.isfinite(latents).all()):
                        raise RuntimeError("The DiT produced NaN/Inf in FP16 (overflow). Please report this.")
                    clock.mark("head")
                    count = 13 if index == 0 else 16
                    saved_mem = [list(t) if isinstance(t, list) else t for t in decoder.mem]
                    try:
                        frames = decoder.decode_video(latents.transpose(1, 2), parallel=parallel_decode[0],
                                                      show_progress_bar=False,
                                                      cond=lq[:, :, :count].to(DTYPE)).transpose(1, 2)
                    except torch.cuda.OutOfMemoryError:
                        if not parallel_decode[0]:
                            raise
                        decoder.mem = saved_mem  # retry this chunk frame by frame, and stay serial
                        parallel_decode[0] = False
                        torch.cuda.empty_cache()
                        frames = decoder.decode_video(latents.transpose(1, 2), parallel=False,
                                                      show_progress_bar=False,
                                                      cond=lq[:, :, :count].to(DTYPE)).transpose(1, 2)
                    video = frames.mul(2).sub(1)
                    del frames, latents
                    reference = lq[:, :, :video.shape[2]]
                    if factor > 1:
                        video = _merge_segment_groups(video, factor, first=index == 0, mode=cfg.temporal_merge,
                                                      sigma=cfg.temporal_gaussian_sigma)
                        reference = _merge_segment_groups(reference, factor, first=index == 0, mode="first",
                                                          sigma=cfg.temporal_gaussian_sigma)
                    video = video[:, :, :max(n_source_frames - state["produced"], 0)]
                    if corrector is not None and video.shape[2]:
                        video = corrector(video, reference[:, :, :video.shape[2]])
                    clock.mark("decode")
                    if video.shape[2]:
                        sink(video[:, :, :, :out_h, :out_w], index)
                    state["produced"] += video.shape[2]
                    clock.mark("output")
                    del video, reference, lq
                    spans_a, spans_b = msg["clock"].spans(), clock.spans()
                    record = {"index": index, "k": msg["k"], "a": spans_a, "b": spans_b,
                              "a_ms": sum(spans_a.values()), "b_ms": sum(spans_b.values()), "t": time.time()}
                    state["chunks"].append(record)
                    if rebalance and two_gpus and state["pending_k"] is None and msg["k"] == split.k:
                        # Balance the two stages from this chunk's per-block GPU times. Chunk 0 (empty caches)
                        # gives a first estimate; later chunks refine it. A move pauses GPU 0 until GPU 1 is idle,
                        # so it is only made when the time saved on the remaining chunks clearly pays for that.
                        block_ms = [spans_a.get(f"block{j}", spans_b.get(f"block{j}", 0.0))
                                    for j in range(NUM_BLOCKS)]
                        fixed_a = record["a_ms"] - sum(block_ms[:msg["k"]])
                        fixed_b = record["b_ms"] - sum(block_ms[msg["k"]:])
                        new_k, new_cost = best_split(fixed_a, fixed_b, block_ms)
                        old_cost = max(record["a_ms"], record["b_ms"])
                        if index >= 1 or not SPLIT_CACHE.get(split_key):
                            SPLIT_CACHE[split_key] = new_k
                        remaining = (n_chunks or 0) - state["sent"] - 1
                        if (new_k != msg["k"] and new_cost < 0.95 * old_cost and state["moves"] < 3
                                and (old_cost - new_cost) * remaining > record["b_ms"]):
                            state["pending_k"] = new_k
                    with cond:
                        state["done"] = index
                        cond.notify_all()
                    if on_chunk is not None:
                        on_chunk(index, record)
        except BaseException as e:  # noqa: BLE001 - handed to stage A, which re-raises it
            fail(e)

    worker = threading.Thread(target=stage_b, daemon=True)
    worker.start()
    t_start = time.time()
    try:
        with torch.inference_mode():
            for index, value, frame_offset, last in stream.projection_chunks(4, TEMPORAL_DOWNSAMPLE):
                if CANCEL.is_set():
                    raise Cancelled()
                if state["error"] is not None:
                    raise state["error"]
                if state["pending_k"] is not None:  # rebalance: let stage B finish its chunk, then move blocks
                    with cond:
                        while state["done"] < state["sent"] and state["error"] is None:
                            cond.wait(timeout=1.0)
                    if state["error"] is not None:
                        raise state["error"]
                    split.move_split(state["pending_k"], caches)
                    state["pending_k"] = None
                    state["moves"] += 1
                clock = _Clock(DEV_A)
                clock.mark("start")
                video = to_unit(value, DEV_A).to(DTYPE)
                conditions = _project_lq(m.LQ_proj_in, video, index, frame_offset)
                del video
                noise = torch.randn(1, cfg.latent_channels, 4, model_h // 8, model_w // 8, generator=generator,
                                    device=DEV_A, dtype=DTYPE)
                clock.mark("lq")
                x, token_shape = m.patchify(noise)
                if token_shape[0] != 4:
                    raise RuntimeError(f"stream chunk {index} has {token_shape[0]} latent frames, expected 4")
                lq_start, lq_stop = max(0, 16 * index - 3), 16 * (index + 1) - 3
                decoder_lq = stream.window(lq_start, lq_stop)
                stream.release_before(lq_stop)
                k_now = split.k
                x = run_blocks(x, 0, k_now, index, token_shape, not last, conditions, clock)
                del conditions
                msg = {"index": index, "k": k_now, "token_shape": token_shape, "last": last, "lq": decoder_lq,
                       "clock": clock}
                if staged:
                    # GPU 0 -> pinned CPU buffers (async, on GPU 0's stream); stage B waits for the event and copies
                    # them to GPU 1 on its own stream. Neither GPU waits for the other's queued work.
                    msg["x"] = torch.empty(x.shape, dtype=x.dtype, pin_memory=True).copy_(x, non_blocking=True)
                    msg["noise"] = torch.empty(noise.shape, dtype=noise.dtype, pin_memory=True).copy_(
                        noise, non_blocking=True)
                    msg["ready"] = torch.cuda.Event()
                    msg["ready"].record()
                elif two_gpus:
                    msg["x"], msg["noise"] = x.to(DEV_B), noise.to(DEV_B)
                else:
                    msg["x"], msg["noise"] = x, noise
                del x, noise
                while True:
                    if state["error"] is not None:
                        raise state["error"]
                    try:
                        handoff.put(msg, timeout=1.0)
                        break
                    except queue.Full:
                        if CANCEL.is_set():
                            raise Cancelled()
                state["sent"] = index
                if last:
                    break
        handoff.put(None)
        worker.join()
        if state["error"] is not None:
            raise state["error"]
    except BaseException:
        fail(Cancelled() if CANCEL.is_set() else RuntimeError("stage A stopped"))
        try:
            handoff.get_nowait()
        except queue.Empty:
            pass
        handoff.put(None)
        worker.join(timeout=120)
        raise
    finally:
        stream.close()
        caches.clear()
        m.LQ_proj_in.clear_cache()
        decoder.clean_mem()
    wall = time.time() - t_start
    return {"wall": wall, "chunks": state["chunks"], "frames": state["produced"], "k": split.k,
            "parallel_decode": parallel_decode[0]}


def summarize(stats, n_chunks):
    """Steady-state numbers from the per-chunk GPU timings (chunk 0 runs with an empty cache, so it is skipped)."""
    chunks = stats["chunks"]
    steady = chunks[1:] if len(chunks) > 2 else chunks
    a_ms = sum(c["a_ms"] for c in steady) / max(len(steady), 1)
    b_ms = sum(c["b_ms"] for c in steady) / max(len(steady), 1)
    parts = {}
    for c in steady:
        for side in ("a", "b"):
            for name, ms in c[side].items():
                key = "DiT blocks" if name.startswith("block") else name
                parts[(side, key)] = parts.get((side, key), 0.0) + ms / len(steady)
    return a_ms, b_ms, parts


# ==== Engine state ====
MASTERS = {}   # mode -> (CPU FP16 master model, config, missing keys)
ACTIVE = {"split": None, "mode": None}
DECODER = None
CONTEXT = None
SPEED = {}     # (mode, temporal compression) -> {"setup": s per MP, "frame": s per (MP x output frame)} at steady state


def default_split(mode):
    # A first guess for a new resolution, from a 1080p Turbo run on 2x T4 (per chunk: LQ projection 1.9 s, 30 blocks
    # 23.5 s, decoder + head 3.8 s): about half the blocks on each GPU. Measured and corrected during the first job at
    # each size, then remembered.
    return 15


def ensure_mode(mode, model_hw=None, temporal_compression=4, report=print):
    """Make `mode` the model on the GPUs (and its split the best one known for this size)."""
    global DECODER
    if mode not in MASTERS:
        report(f"Loading the {mode.title()} model (first use, ~1 min)...")
        BAR.start("load_" + mode, f"Load {mode.title()} model")
        master, cfg, missing = build_master(mode)
        MASTERS[mode] = (master, cfg, missing)
        BAR.finish(f"({len(missing)} weights not in the checkpoint)" if missing else "")
    master, cfg, _ = MASTERS[mode]
    want_k = SPLIT_CACHE.get((mode, *model_hw, temporal_compression), default_split(mode)) if model_hw else \
        default_split(mode)
    if DEV_A == DEV_B:
        want_k = NUM_BLOCKS  # one GPU: no split
    if ACTIVE["mode"] != mode:
        if ACTIVE["split"] is not None:
            ACTIVE["split"] = None
            gc.collect()
            if CUDA:
                for d in {DEV_A, DEV_B}:
                    with torch.cuda.device(d):
                        torch.cuda.empty_cache()
        report(f"Moving the {mode.title()} model to the GPU{'s' if DEV_A != DEV_B else ''}...")
        ACTIVE["split"] = SplitDiT(master, mode, cfg, want_k)
        ACTIVE["mode"] = mode
        gc.collect()
    elif DEV_A != DEV_B and ACTIVE["split"].k != want_k:
        ACTIVE["split"].move_split(want_k, [None] * NUM_BLOCKS)
    if DECODER is None:
        DECODER = build_decoder(cfg)
    return ACTIVE["split"]


# ==== Live progress (polled by the page over plain HTTP; the tunnels can hold back Gradio's event stream) ====
PROGRESS = {"active": False, "fraction": 0.0, "text": "", "started": 0.0, "finished": 0.0, "video": None, "message": "",
            "job": 0}


def report(fraction, text):
    PROGRESS.update(active=True, fraction=float(fraction), text=text)


MODE_LABELS = {"turbo": "⚡ Turbo (fast)", "base": "💎 Base (finest detail, slower)"}
SIZE_CHOICES = ["1080p", "720p", "1440p (2K)", "2x upscale", "3x upscale", "4x upscale"]
MOTION_CHOICES = {"Normal motion (fastest)": 4, "Large motion (2x slower)": 2, "Very large motion (4x slower)": 1}
MAX_MODEL_AREA = 1536 * 2560


def _mode_from_label(label):
    for mode, text in MODE_LABELS.items():
        if text == label:
            return mode
    return available_modes()[0]


def plan_job(path, size_choice, seconds, motion_label):
    info = probe_video(path)
    fps = info["fps"]
    frames = info["frames"]
    if seconds and seconds > 0:
        frames = min(frames, max(1, int(round(seconds * fps)))) if frames else max(1, int(round(seconds * fps)))
    out_w, out_h = output_size(info["width"], info["height"], size_choice)
    model_h, model_w = model_size(out_w, out_h)
    tc = MOTION_CHOICES.get(motion_label, 4)
    expanded = 1 + (target_frame_count(max(frames, 1), tc) - 1) * (TEMPORAL_DOWNSAMPLE // tc)
    chunks = (expanded - 1) // 16 + 1
    return dict(info=info, frames=frames, out=(out_w, out_h), model=(model_h, model_w), tc=tc, chunks=chunks)


def describe_input(path, size_choice, seconds, motion_label, mode_label):
    if not path:
        return "📼 Upload a video to see its size and the output size."
    try:
        p = plan_job(path, size_choice, seconds, motion_label)
    except Exception as e:
        return f"❌ Cannot read this video: {e}"
    info, (out_w, out_h), (model_h, model_w) = p["info"], p["out"], p["model"]
    text = (f"📼 **Input:** {info['width']}×{info['height']} · {info['fps']:.2f} fps · "
            f"{p['frames']} frames ({p['frames'] / info['fps']:.1f} s)  \n"
            f"🖥️ **Output:** {out_w}×{out_h} ({out_w / info['width']:.2f}×) · {p['chunks']} chunks of 16 frames")
    if (model_h, model_w) != (out_h, out_w):
        text += f" · runs at {model_w}×{model_h} (mirror padding, cropped off)"
    if model_h * model_w > MAX_MODEL_AREA:
        text += "  \n⚠️ **Too large for a T4.** Pick 1440p or smaller."
    rate = SPEED.get((_mode_from_label(mode_label), p["tc"]))
    if rate:
        mp = out_w * out_h / 1e6
        text += (f"  \n⏱️ **Estimated time:** ~{_fmt_secs(rate['setup'] * mp + rate['frame'] * mp * p['frames'])} "
                 f"(from your last run)")
    else:
        text += "  \n⏱️ The first video measures the speed; later ones show a time estimate here."
    return text


def upscale_video(path, mode_label, size_choice, seconds, motion_label, color_fix, seed, crf, preset, compare=True):
    """Read, upscale and encode one video. Returns (mp4 path, status text, comparison HTML)."""
    t0 = time.time()
    mode = _mode_from_label(mode_label)
    seed = random.randint(0, 2**31 - 1) if seed is None or int(seed) < 0 else int(seed)
    plan = plan_job(path, size_choice, seconds, motion_label)
    info, (out_w, out_h), (model_h, model_w), tc = plan["info"], plan["out"], plan["model"], plan["tc"]
    if model_h * model_w > MAX_MODEL_AREA:
        raise ValueError(f"{out_w}×{out_h} is too large for a 16 GB T4. Pick 1440p or smaller.")
    report(0.0, "Reading the video...")
    frames = read_frames(path, plan["frames"] if seconds and seconds > 0 else None)
    n = len(frames)
    source_bytes = n * frames[0].nbytes
    if source_bytes > 0.5 * psutil.virtual_memory().available:
        raise ValueError(f"The decoded video needs {_gb(source_bytes)} of RAM. Upscale a shorter part "
                         "(Seconds to upscale).")
    expanded = 1 + (target_frame_count(n, tc) - 1) * (TEMPORAL_DOWNSAMPLE // tc)
    n_chunks = (expanded - 1) // 16 + 1
    fps = info["fps"]
    print(f"\n🎞️ Job {JOB['id']} | {mode.title()} | {info['width']}×{info['height']} → {out_w}×{out_h} "
          f"(model {model_w}×{model_h}) | {n} frames ({n / fps:.1f} s) | {n_chunks} chunks | "
          f"{str(motion_label).split(' (')[0].lower()} | seed {seed}")
    sys.stdout.flush()
    split = ensure_mode(mode, (model_h, model_w), tc, report=lambda t: report(0.0, t))
    stem = f"flashvsr_{mode}_{out_w}x{out_h}_{time.strftime('%Y%m%d_%H%M%S')}"
    raw_path = os.path.join(OUTPUT_DIR, f".{stem}.video.mp4")
    final_path = os.path.join(OUTPUT_DIR, f"{stem}.mp4")
    writer = VideoWriter(raw_path, out_w, out_h, fps, crf=crf, preset=preset)
    # The comparison's 'before': the model's own input frames (bicubic to the output size), encoded alongside.
    before_path = os.path.join(OUTPUT_DIR, f"compare_bicubic_{stem[len('flashvsr_'):]}.mp4") if compare else None
    before_writer = VideoWriter(before_path, out_w, out_h, fps, crf=crf, preset=preset,
                                queue_size=96) if compare else None
    tap = (lambda frame: before_writer.push(frame[None])) if compare else None

    def sink(video, index):
        frames_u8 = video[0].float().add(1).div(2).clamp(0, 1).mul(255).round().to(torch.uint8)
        writer.push(frames_u8.permute(1, 2, 3, 0).contiguous().cpu().numpy())

    def on_chunk(index, record):
        done = index + 1
        BAR.update(done)
        elapsed = time.time() - t_run
        eta = elapsed / done * (n_chunks - done)
        report(0.97 * done / n_chunks, f"Chunk {done}/{n_chunks} · {min(done * 16, n)}/{n} frames · ETA {_fmt_secs(eta)}")

    report(0.01, f"Upscaling {n} frames ({n_chunks} chunks)...")
    BAR.start("upscale", "Upscaling", total=n_chunks, unit="chunks")
    t_run = time.time()
    try:
        stats = run_pipeline(split, DECODER, CONTEXT, lq_chunks(frames, out_w, out_h, model_h, model_w, tap=tap),
                             n_source_frames=n, model_hw=(model_h, model_w), out_hw=(out_h, out_w),
                             temporal_compression=tc, seed=seed, color_fix=bool(color_fix), sink=sink,
                             on_chunk=on_chunk, n_chunks=n_chunks)
    except BaseException:
        BAR.abort("stopped" if CANCEL.is_set() else "failed")
        writer.abort()
        if before_writer is not None:
            before_writer.abort()
        for p in (raw_path, before_path):
            if p and os.path.exists(p):
                os.remove(p)
        raise
    BAR.finish()
    del frames
    report(0.98, "Encoding + adding the audio track...")
    writer.close()
    if before_writer is not None:
        before_writer.close()
    with_audio = mux_audio(raw_path, path, final_path, n / fps)
    elapsed = time.time() - t0
    run_s = stats["wall"]
    a_ms, b_ms, parts = summarize(stats, n_chunks)
    # Timeline: the first chunk carries the one-time costs (new conv shapes, empty caches, the pipeline filling up);
    # after it, one chunk leaves GPU 1 every `interval` seconds: the throughput a long video gets.
    done_at = [c["t"] - t_run for c in stats["chunks"]]
    frames_per_chunk = 16 // (TEMPORAL_DOWNSAMPLE // tc)
    interval = (done_at[-1] - done_at[0]) / (len(done_at) - 1) if len(done_at) > 1 else None
    mp = out_w * out_h / 1e6
    if interval:
        SPEED[(mode, tc)] = {"setup": max(done_at[0] - interval, 0.0) / mp, "frame": interval / frames_per_chunk / mp}
    else:
        SPEED[(mode, tc)] = {"setup": 0.0, "frame": run_s / max(n, 1) / mp}
    fps_out = n / max(run_s, 1e-6)
    gpu_line = ""
    if DEV_A != DEV_B:
        lq = parts.get(("a", "lq"), 0.0) / 1000
        blocks_a = parts.get(("a", "DiT blocks"), 0.0) / 1000
        blocks_b = parts.get(("b", "DiT blocks"), 0.0) / 1000
        dec = parts.get(("b", "decode"), 0.0) / 1000 + parts.get(("b", "head"), 0.0) / 1000
        out_s = parts.get(("b", "output"), 0.0) / 1000
        gpu_line = (f"\n  ⚖️ Per chunk: GPU 0 {a_ms / 1000:.1f}s (LQ {lq:.1f}s + blocks 0-{stats['k'] - 1} "
                    f"{blocks_a:.1f}s) | GPU 1 {b_ms / 1000:.1f}s (blocks {stats['k']}-29 {blocks_b:.1f}s + decoder "
                    f"{dec:.1f}s + output {out_s:.1f}s)")
    peaks = ""
    if CUDA:
        peaks = " | peak VRAM " + ", ".join(f"GPU {i} {_gb(torch.cuda.max_memory_allocated(i))}" for i in range(N_GPU))
        for i in range(N_GPU):
            torch.cuda.reset_peak_memory_stats(i)
    first = stats["chunks"][0]
    timeline = (f"\n  ⏱️ Timeline: first chunk out after {done_at[0]:.1f}s (one-time costs; its GPU time: "
                f"GPU 0 {first['a_ms'] / 1000:.1f}s, GPU 1 {first['b_ms'] / 1000:.1f}s)")
    if interval:
        timeline += (f", then one chunk every {interval:.1f}s = {interval / frames_per_chunk:.2f} s/frame "
                     f"({frames_per_chunk / interval:.2f} fps) for the rest of a long video")
    print(f"  📊 Speed: {n} frames at {out_w}×{out_h} in {_fmt_secs(run_s)} → {fps_out:.2f} fps "
          f"({run_s / max(n, 1):.2f} s/frame, {n / fps / max(run_s, 1e-6):.2f}x real time){peaks}{gpu_line}{timeline}")
    print(f"  💾 {final_path}{'' if with_audio else ' (no audio track)'} | total {_fmt_secs(elapsed)}")
    sys.stdout.flush()
    status = (f"✅ {n} frames upscaled to {out_w}×{out_h} in {_fmt_secs(run_s)} ({fps_out:.2f} fps, "
              f"{run_s / max(n, 1):.2f} s per frame) · {mode.title()} · seed {seed}\n💾 {final_path}")
    html = compare_html(before_path, final_path, out_w, out_h, fps) if compare else COMPARE_IDLE
    return final_path, status, html


# ==== Before / after comparison player ====
COMPARE_IDLE = ('<div class="aiq-cmp-empty">🔍 The before / after comparison appears here when a video is '
                'upscaled.</div>')


def _file_url(path):
    return "gradio_api/file=" + urllib.parse.quote(path, safe="/")  # Gradio's file route (supports seeking)


def compare_html(before_path, after_path, width, height, fps):
    """Wipe comparison: both videos stacked, the 'before' clipped at a draggable divider; HEAD_JS wires it up."""
    if not before_path or not os.path.exists(before_path):
        return COMPARE_IDLE
    return f"""
<div class="aiq-compare" tabindex="0" data-fps="{fps:.6f}">
  <div class="aiq-cmp-stage" style="aspect-ratio:{width}/{height}">
    <video class="aiq-after" src="{_file_url(after_path)}" preload="auto" playsinline></video>
    <div class="aiq-cmp-clip"><video class="aiq-before" src="{_file_url(before_path)}" preload="auto" muted
      playsinline></video></div>
    <div class="aiq-cmp-line"><div class="aiq-cmp-knob">⟷</div></div>
    <span class="aiq-tag aiq-tag-l">BEFORE · bicubic</span>
    <span class="aiq-tag aiq-tag-r">AFTER · FlashVSR+</span>
  </div>
  <div class="aiq-cmp-bar">
    <button class="aiq-btn aiq-play" type="button">▶ Play</button>
    <button class="aiq-btn aiq-prev" type="button" title="Previous frame (←)">⏮</button>
    <button class="aiq-btn aiq-next" type="button" title="Next frame (→)">⏭</button>
    <input class="aiq-seek" type="range" min="0" max="1000" step="1" value="0">
    <span class="aiq-time">0:00 / 0:00</span>
    <button class="aiq-btn aiq-loop" type="button" title="Loop">🔁</button>
    <button class="aiq-btn aiq-mute" type="button" title="Sound">🔊</button>
    <button class="aiq-btn aiq-full" type="button" title="Fullscreen">⛶</button>
  </div>
  <div class="aiq-cmp-hint">Drag on the video to move the divider · Space: play / pause · ← →: one frame ·
    {width}×{height}</div>
</div>"""


# ==== Generation as a background job ====
# The Upscale button only starts a job and returns. The page asks for the job's state every second with short,
# independent requests (queue=False: plain HTTP, no event stream), so a dropped tunnel connection or a reloaded page
# never loses the result.
JOB = {"id": 0, "done": 0, "running": False, "video": None, "status": "Ready.", "compare": None}


def _job_worker(job_id, args):
    try:
        if CUDA:
            torch.cuda.set_device(DEV_A)
        video, status, compare = upscale_video(*args)
    except Cancelled:
        video, status, compare = None, "⏹️ Stopped.", COMPARE_IDLE
    except torch.cuda.OutOfMemoryError:
        import traceback
        traceback.print_exc()
        video, status = None, "❌ Out of GPU memory. Pick a smaller output size (or Turbo instead of Base)."
        compare = COMPARE_IDLE
    except Exception as e:
        import traceback
        traceback.print_exc()
        video, status, compare = None, f"❌ Error: {e}", COMPARE_IDLE
    finally:
        CANCEL.clear()
        gc.collect()
        if CUDA:
            for i in range(N_GPU):
                with torch.cuda.device(i):
                    torch.cuda.empty_cache()
    JOB.update(running=False, done=job_id, video=video, status=status, compare=compare)
    PROGRESS.update(active=False, finished=time.time(), fraction=1.0 if video else PROGRESS["fraction"],
                    video=os.path.basename(video) if video else None, message=status.splitlines()[0], job=job_id)


# ==== Startup ====
T_LOAD = time.time()
print("=" * 70)
print("🎞️ FlashVSR+ Video Upscaler | AIQUEST Academy")
print(f"⚡ GPUs: {N_GPU} x {_props[0].name} ({_gb(_props[0].total_memory)} each, SM {_props[0].major}.{_props[0].minor}) | "
      f"RAM: {_gb(psutil.virtual_memory().total)} ({_gb(psutil.virtual_memory().available)} free)")
if N_GPU < 2:
    print("⚠️ Only one GPU: everything runs on GPU 0 (about half the speed). Use Settings → Accelerator → GPU T4 x2.")
print("=" * 70)
sys.stdout.flush()

# Triton only has a tensor-core path for its dot products from Ampere (SM 8.0) on. Measured on a T4 (SM 7.5): every
# tile setting was correct but 15-20x slower than PyTorch's fused attention, and compiling them cost 3 minutes.
_use_triton = _props[0].major >= 8
BAR.start("tune", "Tune sparse attention" + (" (first start compiles ~8 kernels)" if _use_triton else ""))
_candidates = ([("triton", c) for c in TRITON_CONFIGS] if _use_triton else []) + \
    [("torch", "varlen"), ("torch", "sdpa"), ("torch", "sdpa_nomask"), ("torch", "matmul")]
_rows = tune_attention(DEV_A, _candidates)
torch.cuda.empty_cache()
BAR.finish()
print("  ⚡ Sparse attention on GPU 0" + ("" if _use_triton else
                                        f" (Triton skipped: no tensor-core path on SM {_props[0].major}.{_props[0].minor})")
      + ":")
for _kind, _config, _ms, _note in _rows:
    print(f"     {'✅' if _ms is not None else '❌'} {_attn_label(_kind, _config):<38} "
          f"{f'{_ms:7.1f} ms' if _ms is not None else '      -  '}  {_note}")
_best = ATTN[dkey(DEV_A)]
if DEV_B != DEV_A:  # same GPU model: check the winner (and the PyTorch path) on GPU 1 instead of tuning again
    _rows_b = tune_attention(DEV_B, [_best] + ([("torch", "sdpa")] if _best != ("torch", "sdpa") else []))
    if ATTN[dkey(DEV_B)] != _best:
        print(f"  ⚠️ GPU 1 could not use {_attn_label(*_best)}; it uses {_attn_label(*ATTN[dkey(DEV_B)])}")
    _b = next((r for r in _rows_b if (r[0], r[1]) == ATTN[dkey(DEV_B)]), None)
    print(f"  ⚡ Using {_attn_label(*_best)} on GPU 0"
          + (f"; GPU 1: {_attn_label(*ATTN[dkey(DEV_B)])} ({_b[2]:.1f} ms)" if _b and _b[2] else ""))
else:
    print(f"  ⚡ Using {_attn_label(*_best)}")

MODES = available_modes()
if not MODES:
    raise SystemExit(f"❌ No FlashVSR+ weights in {MODELS_DIR}. Run Step 1 first.")
CONTEXT = _load_context(os.path.join(REPO_DIR, "prompt_tensor", "posi_prompt.pth"))
ensure_mode(MODES[0])
_master_missing = MASTERS[MODES[0]][2]
if _master_missing:
    print(f"  ⚠️ {len(_master_missing)} weights are not in the checkpoint and keep their initial values "
          f"(same as the official loader): {', '.join(_master_missing[:4])}{' ...' if len(_master_missing) > 4 else ''}")

# Warm-up: compiles the kernels and picks the cuDNN algorithms, so the first real video runs at full speed.
BAR.start("warmup", "Warm-up")
_wh, _ww = 256, 384
_frames = [np.full((_wh, _ww, 3), 64 + (i * 5) % 128, dtype=np.uint8) for i in range(33)]
run_pipeline(ACTIVE["split"], DECODER, CONTEXT, lq_chunks(_frames, _ww, _wh, _wh, _ww), n_source_frames=33,
             model_hw=(_wh, _ww), out_hw=(_wh, _ww), temporal_compression=4, seed=0, color_fix=False,
             sink=lambda video, index: None, rebalance=False)
del _frames
BAR.finish()
for _i in range(N_GPU):
    _free, _total = torch.cuda.mem_get_info(_i)
    print(f"  💾 GPU {_i}: {_gb(_total - _free)} / {_gb(_total)} in use")
print(f"✅ Engine ready in {_fmt_secs(time.time() - T_LOAD)}")
sys.stdout.flush()

import gradio as gr  # noqa: E402


def start_upscale(path, mode_label, size_choice, seconds, motion_label, color_fix, seed, crf, preset, compare):
    if JOB["running"]:
        return gr.skip(), "⏳ A video is already being upscaled. It appears here as soon as it is ready.", gr.skip()
    if not path or not os.path.exists(str(path)):
        return gr.skip(), "❌ Upload a video first.", gr.skip()
    args = (path, mode_label, size_choice, seconds, motion_label, color_fix, seed, crf, preset, bool(compare))
    JOB.update(id=JOB["id"] + 1, running=True)
    PROGRESS.update(active=True, fraction=0.0, text="Starting...", started=time.time(), finished=0.0, video=None,
                    message="")
    threading.Thread(target=_job_worker, args=(JOB["id"], args), daemon=True).start()
    return (None, "🚀 Upscaling... the progress bar above updates live, and the video appears here when it is ready.",
            COMPARE_IDLE)


# The progress bar is one fixed element that the page script (HEAD_JS) fills from /aiq_progress every second: only
# its text and width change, so nothing is re-rendered and nothing blinks.
PROGRESS_SHELL = ('<div class="aiq-progress" data-live="1"><div class="aiq-progress-top"><span><span class="aiq-ptext">'
                  '💤 Ready. Upload a video and press Upscale.</span><a class="aiq-plink" target="_blank" '
                  'style="display:none"> · ▶ open / download</a></span><span class="aiq-pct"></span></div>'
                  '<div class="aiq-track"><div class="aiq-fill" style="width:0%"></div></div></div>')


def deliver_job():
    """The finished job's video, status and comparison player. The page script clicks the hidden Deliver button once
    per finished job it has not shown yet (also right after a page reload), so Gradio never polls and no visible
    component is touched while a job runs."""
    return JOB["video"], JOB["status"], JOB["compare"] or COMPARE_IDLE


def stop_upscale():
    if not JOB["running"]:
        return "Nothing is running."
    CANCEL.set()
    return "⏹️ Stopping after the current chunk..."


def _latest_video():
    files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "flashvsr_*.mp4")), key=os.path.getmtime)
    return files[-1] if files else None


def load_latest_video():
    path = _latest_video()
    if not path:
        return None, f"No video in {OUTPUT_DIR} yet.", COMPARE_IDLE
    before = os.path.join(OUTPUT_DIR, "compare_bicubic_" + os.path.basename(path)[len("flashvsr_"):])
    html = COMPARE_IDLE
    if os.path.exists(before):
        info = probe_video(path)
        html = compare_html(before, path, info["width"], info["height"], info["fps"])
    return path, f"📁 Latest video: {path}", html


# ==== Gradio UI (AIQUEST Academy branding) ====
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.status-line { text-align: center; color: #6b7280; font-size: 13px; margin: -8px 0 12px 0; }
.brand-footer { text-align: center; color: #6b7280; font-size: 12px; margin-top: 24px; }
.plan-box { font-size: 13px !important; padding: 8px 12px !important; border-left: 3px solid #764ba2; background: rgba(102,126,234,0.06); border-radius: 4px; }
.plan-box p { margin: 0 !important; }
.aiq-progress { padding: 12px 14px; border-radius: 12px; background: rgba(102,126,234,0.07); border: 1px solid rgba(102,126,234,0.25); }
.aiq-progress-top { display: flex; justify-content: space-between; gap: 12px; font-size: 13px; margin-bottom: 8px; }
.aiq-pct { font-weight: 700; color: #764ba2; }
.aiq-track { height: 10px; background: rgba(118,75,162,0.15); border-radius: 6px; overflow: hidden; }
.aiq-fill { height: 100%; border-radius: 6px; transition: width .8s ease; }
.aiq-cmp-empty { text-align: center; color: #6b7280; font-size: 13px; padding: 18px; border: 1px dashed rgba(118,75,162,0.35); border-radius: 12px; }
.aiq-compare { outline: none; }
.aiq-cmp-stage { position: relative; width: 100%; max-height: 80vh; background: #000; border-radius: 12px; overflow: hidden; cursor: ew-resize; touch-action: none; user-select: none; -webkit-user-select: none; }
.aiq-cmp-stage video { position: absolute; inset: 0; width: 100%; height: 100%; object-fit: contain; display: block; pointer-events: none; }
.aiq-cmp-clip { position: absolute; inset: 0; clip-path: inset(0 50% 0 0); }
.aiq-cmp-line { position: absolute; top: 0; bottom: 0; left: 50%; width: 2px; margin-left: -1px; background: #fff; box-shadow: 0 0 6px rgba(0,0,0,.6); pointer-events: none; }
.aiq-cmp-knob { position: absolute; top: 50%; left: 50%; transform: translate(-50%, -50%); width: 36px; height: 36px; border-radius: 50%; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: #fff; display: flex; align-items: center; justify-content: center; font-size: 17px; box-shadow: 0 2px 10px rgba(0,0,0,.45); }
.aiq-tag { position: absolute; top: 10px; padding: 3px 9px; border-radius: 6px; font-size: 11px; font-weight: 700; letter-spacing: .04em; color: #fff; background: rgba(0,0,0,.55); pointer-events: none; }
.aiq-tag-l { left: 10px; } .aiq-tag-r { right: 10px; background: rgba(118,75,162,.85); }
.aiq-cmp-bar { display: flex; align-items: center; gap: 8px; margin-top: 10px; flex-wrap: wrap; }
.aiq-seek { flex: 1; min-width: 140px; accent-color: #764ba2; }
.aiq-btn { border: none; border-radius: 9px; padding: 7px 12px; font-weight: 600; font-size: 13px; cursor: pointer; color: #fff; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); }
.aiq-btn.aiq-off { background: #9ca3af; }
.aiq-play { min-width: 92px; }
.aiq-time { font-size: 12px; color: #6b7280; min-width: 92px; text-align: right; font-variant-numeric: tabular-nums; }
.aiq-cmp-hint { font-size: 12px; color: #6b7280; margin-top: 6px; text-align: center; }
#aiq-deliver { display: none !important; }
"""

# Wires up every .aiq-compare player Gradio puts on the page (gr.HTML does not run scripts, so this lives in head=).
# The AFTER video is the clock; the BEFORE video follows it (re-aligned on pause, seek and any drift over 1/10 s).
HEAD_JS = """
<script>
(function () {
  function fmt(t) {
    if (!isFinite(t)) t = 0;
    return Math.floor(t / 60) + ":" + String(Math.floor(t % 60)).padStart(2, "0");
  }
  function init(root) {
    root.dataset.ready = "1";
    var stage = root.querySelector(".aiq-cmp-stage"), clip = root.querySelector(".aiq-cmp-clip");
    var line = root.querySelector(".aiq-cmp-line"), after = root.querySelector("video.aiq-after");
    var before = root.querySelector("video.aiq-before"), play = root.querySelector(".aiq-play");
    var seek = root.querySelector(".aiq-seek"), time = root.querySelector(".aiq-time");
    var loop = root.querySelector(".aiq-loop"), mute = root.querySelector(".aiq-mute");
    var fps = parseFloat(root.dataset.fps) || 30, dragging = false, seeking = false;
    before.muted = true;
    function setPos(p) {
      p = Math.max(0, Math.min(100, p));
      clip.style.clipPath = "inset(0 " + (100 - p) + "% 0 0)";
      line.style.left = p + "%";
    }
    function fromEvent(e) {
      var r = stage.getBoundingClientRect();
      return (e.clientX - r.left) / r.width * 100;
    }
    function align() { if (Math.abs(before.currentTime - after.currentTime) > 0.001) before.currentTime = after.currentTime; }
    function showState() { play.textContent = after.paused ? "▶ Play" : "⏸ Pause"; }
    function toggle() {
      if (after.paused) {
        if (after.ended || after.currentTime >= (after.duration || 0) - 0.01) { after.currentTime = 0; }
        align();
        var a = after.play(), b = before.play();
        if (a && a.catch) a.catch(function () {});
        if (b && b.catch) b.catch(function () {});
      } else {
        after.pause(); before.pause(); align();
      }
    }
    function step(dir) {
      after.pause(); before.pause();
      var t = Math.max(0, Math.min((after.duration || 0), after.currentTime + dir / fps));
      after.currentTime = t; before.currentTime = t;
    }
    setPos(50);
    stage.addEventListener("pointerdown", function (e) {
      dragging = true; stage.setPointerCapture(e.pointerId); setPos(fromEvent(e)); e.preventDefault();
      root.focus({ preventScroll: true });
    });
    stage.addEventListener("pointermove", function (e) { if (dragging) setPos(fromEvent(e)); });
    ["pointerup", "pointercancel"].forEach(function (n) { stage.addEventListener(n, function () { dragging = false; }); });
    play.addEventListener("click", toggle);
    root.querySelector(".aiq-prev").addEventListener("click", function () { step(-1); });
    root.querySelector(".aiq-next").addEventListener("click", function () { step(1); });
    loop.addEventListener("click", function () {
      after.loop = before.loop = !after.loop; loop.classList.toggle("aiq-off", !after.loop);
    });
    mute.addEventListener("click", function () {
      after.muted = !after.muted; mute.textContent = after.muted ? "🔇" : "🔊";
    });
    root.querySelector(".aiq-full").addEventListener("click", function () {
      if (document.fullscreenElement) { document.exitFullscreen(); }
      else if (stage.requestFullscreen) { stage.requestFullscreen(); }
    });
    seek.addEventListener("input", function () {
      seeking = true;
      var t = (after.duration || 0) * seek.value / 1000;
      after.currentTime = t; before.currentTime = t;
    });
    seek.addEventListener("change", function () { seeking = false; });
    after.addEventListener("play", showState);
    after.addEventListener("pause", function () { before.pause(); align(); showState(); });
    after.addEventListener("ended", function () { before.pause(); showState(); });
    after.addEventListener("seeked", align);
    root.addEventListener("keydown", function (e) {
      if (e.code === "Space") { toggle(); e.preventDefault(); }
      else if (e.code === "ArrowLeft") { step(-1); e.preventDefault(); }
      else if (e.code === "ArrowRight") { step(1); e.preventDefault(); }
    });
    loop.classList.add("aiq-off");
    (function tick() {
      if (!root.isConnected) return;
      if (!after.paused && !before.paused && Math.abs(before.currentTime - after.currentTime) > 0.1) align();
      if (!seeking && after.duration) seek.value = Math.round(after.currentTime / after.duration * 1000);
      var label = fmt(after.currentTime) + " / " + fmt(after.duration);
      if (time.textContent !== label) time.textContent = label;
      requestAnimationFrame(tick);
    })();
  }
  setInterval(function () {
    document.querySelectorAll(".aiq-compare:not([data-ready])").forEach(init);
  }, 400);

  function secs(t) {
    t = Math.max(0, Math.round(t || 0));
    return t >= 60 ? Math.floor(t / 60) + "m" + String(t % 60).padStart(2, "0") + "s" : t + "s";
  }
  var COLORS = { run: "linear-gradient(135deg,#667eea 0%,#764ba2 100%)", ok: "linear-gradient(135deg,#10b981,#059669)",
                 bad: "linear-gradient(135deg,#ef4444,#b91c1c)" };
  function renderProgress(el, p) {
    var label = el.querySelector(".aiq-ptext"), link = el.querySelector(".aiq-plink");
    var pct = el.querySelector(".aiq-pct"), fill = el.querySelector(".aiq-fill");
    var frac = Math.max(0, Math.min(1, p.fraction || 0)), text, color = COLORS.run, href = null;
    if (p.active) {
      text = "⏳ " + (p.text || "Working...") + " · " + secs(p.elapsed);
    } else if (p.finished) {
      var ok = !!p.video;
      text = p.message || (ok ? "Done" : "Stopped");
      color = ok ? COLORS.ok : COLORS.bad;
      if (ok) { frac = 1; href = "aiq_video?t=" + Math.floor(p.finished); }
    } else {
      text = "💤 Ready. Upload a video and press Upscale."; frac = 0;
    }
    if (label.textContent !== text) label.textContent = text;
    if (href) { if (link.getAttribute("href") !== href) link.setAttribute("href", href); link.style.display = ""; }
    else if (link.style.display !== "none") link.style.display = "none";
    var width = (frac * 100).toFixed(1) + "%";
    if (fill.style.width !== width) fill.style.width = width;
    if (fill.dataset.color !== color) { fill.style.background = color; fill.dataset.color = color; }
    var pctText = (p.active || p.finished) ? Math.round(frac * 100) + "%" : "";
    if (pct.textContent !== pctText) pct.textContent = pctText;
  }
  var busy = false, delivered = 0;
  setInterval(function () {
    var el = document.querySelector(".aiq-progress[data-live]");
    if (!el || busy) return;
    busy = true;
    fetch("aiq_progress", { cache: "no-store" }).then(function (r) { return r.json(); })
      .then(function (p) {
        renderProgress(el, p);
        var button = document.getElementById("aiq-deliver");
        if (!p.active && p.job && p.job !== delivered && button) { delivered = p.job; button.click(); }
      }).catch(function () {})
      .then(function () { busy = false; });
  }, 1000);
})();
</script>
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">🎞️ FlashVSR+ Video Upscaler</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; Kaggle GPU T4 x2 Edition</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""
_attn_kinds = {kind for kind, _ in ATTN.values()}
_mode_text = ("DiT split across both GPUs, decoder on GPU 1" if DEV_A != DEV_B else "everything on GPU 0")
STATUS_HTML = (f'<div class="status-line">⚡ FP16 on {N_GPU} x {_props[0].name} &nbsp;|&nbsp; {_mode_text} '
               f'&nbsp;|&nbsp; {"Triton" if _attn_kinds == {"triton"} else "PyTorch"} sparse attention '
               f'&nbsp;|&nbsp; one-step diffusion, streamed in 16-frame chunks</div>')
FOOTER_HTML = ('<div class="brand-footer">⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · '
               '© All rights reserved</div>')
EXAMPLES = [[p] for p in (os.path.join(REPO_DIR, "examples", "case_1.mp4"),
                          os.path.join(REPO_DIR, "examples", "case_2.mp4")) if os.path.exists(p)]

print("🖥️ Building the UI...")
sys.stdout.flush()
with gr.Blocks(theme=gr.themes.Soft(), css=CSS, head=HEAD_JS, title="FlashVSR+ Video Upscaler | AIQUEST Academy",
               analytics_enabled=False) as demo:
    gr.HTML(BRAND_HTML)
    gr.HTML(STATUS_HTML)
    gr.Markdown("💡 **Tips:** FlashVSR+ is trained for **4x** upscaling of low-resolution video and works well from 2x "
                "to 4x. Try the first few seconds first (**Seconds to upscale**), then run the whole clip. "
                "**Turbo** is the fast model; **Base** keeps a little more fine detail. "
                "The audio track is copied to the result.")
    with gr.Row():
        with gr.Column(scale=3):
            video_in = gr.Video(label="🎞️ Input video", sources=["upload"])
            plan_md = gr.Markdown(describe_input(None, SIZE_CHOICES[0], 0, None, None), elem_classes=["plan-box"])
        with gr.Column(scale=2):
            mode_radio = gr.Radio([MODE_LABELS[m] for m in MODES], value=MODE_LABELS[MODES[0]], label="🧠 Model")
            size_dd = gr.Dropdown(SIZE_CHOICES, value=SIZE_CHOICES[0], label="🖥️ Output size",
                                  info="720p / 1080p / 1440p = the short side of the result")
            seconds_slider = gr.Slider(0, 600, value=0, step=1, label="⏱️ Seconds to upscale (0 = the whole video)")
            motion_dd = gr.Dropdown(list(MOTION_CHOICES), value=next(iter(MOTION_CHOICES)), label="🏃 Motion",
                                    info="Fast action or camera moves: pick Large motion if details smear")
            with gr.Accordion("⚙️ Advanced", open=False):
                color_cb = gr.Checkbox(value=False, label="🎨 Color fix (match the input's colors)")
                compare_cb = gr.Checkbox(value=True, label="🔍 Before / after comparison (also saves the bicubic "
                                                           "version, a few % slower)")
                seed_input = gr.Number(value=-1, precision=0, label="🎲 Seed (-1 = random)")
                crf_slider = gr.Slider(10, 28, value=17, step=1, label="📦 Video quality (CRF, lower = better)")
                preset_dd = gr.Dropdown(["ultrafast", "superfast", "veryfast", "faster", "fast", "medium"],
                                        value="veryfast", label="Encoder preset")

    with gr.Row():
        gen_btn = gr.Button("🚀 Upscale Video", variant="primary", size="lg")
        stop_btn = gr.Button("🛑 Stop", size="lg", elem_id="stop-btn")
        clear_btn = gr.Button("🗑️ Clear", size="lg", elem_id="clear-btn")
    progress_html = gr.HTML(PROGRESS_SHELL)
    video_out = gr.Video(label="✨ Upscaled video", interactive=False)
    status_box = gr.Textbox(label="ℹ️ Status", lines=2, interactive=False)
    with gr.Group():
        gr.Markdown("### 🔍 Before / after\nLeft of the divider: the input resized with bicubic to the same size. "
                    "Right: FlashVSR+. Both play in sync.")
        compare_html_box = gr.HTML(COMPARE_IDLE)
    latest_btn = gr.Button("📂 Load latest video (if the page lost its connection)")
    deliver_btn = gr.Button("Deliver", elem_id="aiq-deliver")  # hidden; clicked by the page script
    if EXAMPLES:
        gr.Examples(EXAMPLES, inputs=[video_in], label="✨ Example clips from the FlashVSR+ repo (click, then Upscale)")
    gr.HTML(FOOTER_HTML)

    plan_inputs = [video_in, size_dd, seconds_slider, motion_dd, mode_radio]
    for trigger in (video_in, size_dd, seconds_slider, motion_dd, mode_radio):
        trigger.change(describe_input, inputs=plan_inputs, outputs=plan_md, queue=False, show_progress="hidden")
    gen_btn.click(start_upscale,
                  inputs=[video_in, mode_radio, size_dd, seconds_slider, motion_dd, color_cb, seed_input, crf_slider,
                          preset_dd, compare_cb],
                  outputs=[video_out, status_box, compare_html_box], queue=False, show_progress="hidden")
    deliver_btn.click(deliver_job, outputs=[video_out, status_box, compare_html_box], queue=False,
                      show_progress="hidden")
    stop_btn.click(stop_upscale, outputs=status_box, queue=False)
    latest_btn.click(load_latest_video, outputs=[video_out, status_box, compare_html_box], queue=False)
    clear_btn.click(lambda: (None, None, "Ready.", COMPARE_IDLE),
                    outputs=[video_in, video_out, status_box, compare_html_box], queue=False)


# ==== Cloudflare quick tunnel: a second public link in case the Gradio share link fails ====
# cloudflared tries QUIC (UDP) first, which Kaggle's network often blocks: hence --protocol http2. A browser that asks
# for the random *.trycloudflare.com name before it exists caches the "not found" answer for minutes, so the link is
# only shown as ready once cloudflared has registered AND the name resolves.
def _dns_published(host):
    """Ask Cloudflare's DNS-over-HTTPS directly (no local resolver caches a negative answer). True / False / None."""
    try:
        req = urllib.request.Request(f"https://cloudflare-dns.com/dns-query?name={host}&type=A",
                                     headers={"accept": "application/dns-json"})
        return bool(json.loads(urllib.request.urlopen(req, timeout=5).read()).get("Answer"))
    except Exception:
        return None


def start_cloudflare_tunnel(port):
    import stat
    found = {"url": None, "registered": False, "ready": False}
    binary = "/tmp/cloudflared"
    try:
        if not os.path.exists(binary):
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
            os.chmod(binary, os.stat(binary).st_mode | stat.S_IEXEC)
        tunnel = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate",
                                   "--protocol", "http2"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        import atexit
        atexit.register(tunnel.kill)
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel unavailable: {e}")
        return found

    def _watch():
        for line in tunnel.stdout:  # keep draining so cloudflared never blocks on a full pipe
            match = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", line)
            if match and not found["url"]:
                found["url"] = match.group(0)
            if "Registered tunnel connection" in line:
                found["registered"] = True

    def _wait_ready():
        deadline, registered_at = time.time() + 300, None
        while time.time() < deadline:
            if found["registered"] and registered_at is None:
                registered_at = time.time()
            published = _dns_published(found["url"].split("//", 1)[1]) if found["url"] and registered_at else False
            if published or (published is None and time.time() - registered_at > 15):
                found["ready"] = True
                if PRINTED_LINKS.is_set():
                    print(f"\n✅ Cloudflare link is ready: {found['url']}\n")
                    sys.stdout.flush()
                return
            time.sleep(3)
        print("⚠️ The Cloudflare link did not come up within 5 minutes; use the Gradio share link.")

    threading.Thread(target=_watch, daemon=True).start()
    threading.Thread(target=_wait_ready, daemon=True).start()
    return found


PRINTED_LINKS = threading.Event()
print("🌐 Starting the public links (Cloudflare + Gradio share)...")
sys.stdout.flush()
cf = start_cloudflare_tunnel(SERVER_PORT)
demo.queue(max_size=20)
demo.launch(server_name="127.0.0.1", server_port=SERVER_PORT, share=True, inline=False, show_error=True,
            prevent_thread_lock=True, quiet=True, ssr_mode=False, show_api=False, max_threads=8,
            allowed_paths=[OUTPUT_DIR, os.path.join(REPO_DIR, "examples")])
from fastapi.responses import FileResponse, JSONResponse  # noqa: E402
from fastapi.routing import APIRoute  # noqa: E402


def _progress_route():
    end = PROGRESS["finished"] or time.time()
    elapsed = end - PROGRESS["started"] if PROGRESS["started"] else 0.0
    return JSONResponse({**PROGRESS, "elapsed": elapsed}, headers={"Cache-Control": "no-store"})


def _video_route():
    """The newest video as a plain file (plays in the browser, with a download option): no Gradio involved."""
    path = _latest_video()
    if not path:
        return JSONResponse({"error": "no video yet"}, status_code=404)
    name = os.path.basename(path)
    return FileResponse(path, media_type="video/mp4",
                        headers={"Content-Disposition": f'inline; filename="{name}"', "Cache-Control": "no-store"})


for _path, _route in (("/aiq_progress", _progress_route), ("/aiq_video", _video_route)):
    demo.app.router.routes.insert(0, APIRoute(_path, _route, methods=["GET"]))  # ahead of Gradio's routes
for _ in range(60):  # give the tunnel up to ~30 s before printing the links
    if cf["ready"]:
        break
    time.sleep(0.5)

print("\n" + "=" * 70)
print("🎞️ FlashVSR+ is LIVE!")
print("=" * 70)
if cf["ready"]:
    print(f"🌐 Cloudflare tunnel: {cf['url']}")
else:
    print(f"🌐 Cloudflare tunnel: ⏳ warming up{' (' + cf['url'] + ')' if cf['url'] else ''} - wait for the "
          f"'✅ Cloudflare link is ready' line before opening it")
print(f"🔗 Gradio share:      {getattr(demo, 'share_url', None) or '(unavailable, use the Cloudflare link)'}")
print(f"💾 Videos are saved to {OUTPUT_DIR} (Kaggle Output panel)")
print("🎞️ Latest video without the UI: <link>/aiq_video")
print("=" * 70 + "\n")
PRINTED_LINKS.set()
sys.stdout.flush()
while True:  # keep the server process alive; job progress streams to the notebook cell
    time.sleep(3600)
'''

with open("run_flashvsr.py", "w", encoding="utf-8") as f:
    f.write(RUN_SCRIPT_CODE)
print("✅ run_flashvsr.py written.")

if os.path.exists("flashvsr_settings.json"):  # Step 1's paths, for a Step 2 re-run after a kernel restart
    with open("flashvsr_settings.json") as f:
        for key, value in json.load(f).items():
            os.environ.setdefault(key, value)

# Library noise that is not an error; everything else (including tracebacks) is shown.
IGNORE = ("absl::InitializeLog", "Unable to register cu", "computation placer already registered", "MessageFactory",
          "GetPrototype", "Attempting to register factory", "UserWarning", "FutureWarning", "DeprecationWarning",
          "warnings.warn", "TypedStorage is deprecated")
env = dict(os.environ, PYTHONWARNINGS="ignore", TQDM_DISABLE="1", HF_HUB_DISABLE_PROGRESS_BARS="1",
           GRADIO_ANALYTICS_ENABLED="False")

print("🚀 Starting the FlashVSR+ engine...\n")
import psutil


def stop_old_engine():
    # A previous run's engine (or one left behind by a kernel restart) still holds both GPUs and RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_flashvsr.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} leftover engine process(es) from an earlier run: GPU memory and RAM are free again")


stop_old_engine()

# Step 1's downloads leave freed buffers in this kernel process: hand them back to the OS before the engine starts.
import ctypes
import gc
gc.collect()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass


def _die_with_kernel():
    # Linux: the engine gets SIGKILL when this notebook kernel dies or restarts (no orphan holding the GPUs).
    try:
        import signal
        ctypes.CDLL("libc.so.6").prctl(1, signal.SIGKILL)  # PR_SET_PDEATHSIG
    except Exception:
        pass


proc = subprocess.Popen([sys.executable, "-u", "run_flashvsr.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        env=env, preexec_fn=_die_with_kernel)
import threading
_peak = {"ram": 0.0, "engine": 0.0}


def _watch_ram():  # peak RAM while the engine runs, reported if it dies
    engine = psutil.Process(proc.pid)
    while proc.poll() is None:
        try:
            vm = psutil.virtual_memory()
            _peak["ram"] = max(_peak["ram"], (vm.total - vm.available) / 1024**3)
            _peak["engine"] = max(_peak["engine"], engine.memory_info().rss / 1024**3)
        except Exception:
            pass
        time.sleep(0.5)


threading.Thread(target=_watch_ram, daemon=True).start()
# Read raw bytes so carriage returns survive: the engine redraws its live progress line in place, and a text-mode
# pipe would turn every redraw into a new line.
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
pending = ""
try:
    while True:
        chunk = os.read(proc.stdout.fileno(), 65536)
        if not chunk:
            break
        pending += decoder.decode(chunk)
        parts = re.split(r"(?<=[\r\n])", pending)
        pending = parts.pop()  # text after the last line break / carriage return
        out = "".join(part for part in parts if not any(pattern in part for pattern in IGNORE))
        if pending and not any(pattern in pending for pattern in IGNORE):
            out, pending = out + pending, ""  # show the live progress line right away
        if out:
            sys.stdout.write(out)
            sys.stdout.flush()
    sys.stdout.write(pending)
    rc = proc.wait()
    if rc != 0:
        import signal
        how = f"killed by {signal.Signals(-rc).name}" if rc < 0 else f"exit code {rc}"
        hint = (" - the system killed it, which almost always means it ran out of RAM" if rc == -9 else
                " - a crash inside a native library (see the traceback above)" if rc in (-11, -6) else "")
        print(f"\n❌ The engine stopped ({how}){hint}.\n   Peak RAM while it ran: {_peak['ram']:.1f} GB in use of "
              f"{psutil.virtual_memory().total / 1024**3:.1f} GB (engine process {_peak['engine']:.1f} GB).")
except KeyboardInterrupt:
    proc.kill()  # frees the GPUs and RAM right away
    proc.wait()
    print("🛑 Engine stopped. Re-run this cell to start it again.")

---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---